# NB12 — RQ2 Power Analysis

NB06 returned a **null** for RQ2: 8 of 36 (weapon × outcome × lag) Dumitrescu-Hurlin
cells were nominally significant, but every one failed placebo falsification
(placebo significance rates 0.20–0.50 against a 0.15 threshold). The standard
reviewer attack on a null is *"you just lacked power."* This notebook pre-empts it:

| Section | Content |
|---|---|
| 0 | Setup — rebuild the NB06 panel from `src.rq2_panel` |
| 1 | Simulation-based power curve: what effect sizes CAN the pipeline detect? |
| 2 | Permutation null: is 8/36 significant cells more than chance? |
| 2b | Per-cell permutation reconciliation of the count-level excess |
| 3 | Diagnosis of the systematic negative-Z artifact at lags 2–3 |
| 4 | Sanity checks |
| 5 | Headline findings |

The panel construction is refactored into `src/rq2_panel.py` (replicating NB06
Sections 0–1 exactly — NB06 itself is unchanged as the historical record), and the
DH test is `src.stats_panel.dumitrescu_hurlin_fast` throughout. All randomness is
seeded through `np.random.default_rng([SEED, task_id])` children so re-runs
reproduce exactly; heavy loops are parallelized with joblib and cached to
`data/interim/`.

## Section 0 — Setup

Rebuild the RQ2 panel, extract per-country numpy arrays, define constants, and
time the DH test to bound total runtime (with an automatic reduction of
`N_SIM`/`N_PERM` if the estimate exceeds 30 serial minutes).

In [1]:
import sys, os, time
from pathlib import Path
sys.path.insert(0, '..')

from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from joblib import Parallel, delayed

from src.config import SEED, DATA_DIR, FIGURES_DIR, TABLES_DIR
from src.rq2_panel import (build_rq2_panel, extract_country_data,
                           WEAPON_CLASSES, OUTCOMES)
from src.stats_panel import dumitrescu_hurlin_fast

# loky workers import src.stats_panel by reference — expose the project root
PROJECT_ROOT = Path("..").resolve()
os.environ["PYTHONPATH"] = str(PROJECT_ROOT) + os.pathsep + os.environ.get("PYTHONPATH", "")

FIG_DIR = FIGURES_DIR / "nb12"
TBL_DIR = TABLES_DIR / "nb12"
CACHE_DIR = DATA_DIR / "interim"
for d in (FIG_DIR, TBL_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

panel = build_rq2_panel()
n_countries = panel["iso3"].nunique()
assert n_countries == 192, f"expected 192 countries, got {n_countries}"

WEAPON_CLASS_NAMES = list(WEAPON_CLASSES)          # AIR, MISSILES, NAVAL, GROUND
TREAT_COLS = [f"d_log_tiv_{c}" for c in WEAPON_CLASS_NAMES]
OUT_COLS = [f"d_log_{o}" for o in OUTCOMES]
country_data = extract_country_data(panel, TREAT_COLS + OUT_COLS)

LAGS = [1, 2, 3]
ALPHA = 0.05
PLACEBO_THRESHOLD = 0.15
OBSERVED_SIG_CELLS = 8      # NB06 Section 3: nominally significant cells, all lag 1
MAX_OBSERVED_CCF = 0.04     # NB06 Section 2: largest |cross-correlation|
N_PERM = 200
N_SIM = 300
N_NOISE = 100
INJECTION_CELLS = [("GROUND", "part_n_minor"), ("MISSILES", "part_n_minor")]
DELTAS = [0.0, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30]   # 0.0 = rejection-at-zero check
GRID_CELLS = list(product(WEAPON_CLASS_NAMES, OUTCOMES, LAGS))

print(f"Panel: {panel.shape}, {n_countries} countries, "
      f"{panel['year'].min()}–{panel['year'].max()}")
print(f"Grid: {len(GRID_CELLS)} cells | injection cells: {INJECTION_CELLS}")

# Runtime guard: time the DH test, estimate the full budget
t0 = time.perf_counter()
for _ in range(10):
    dumitrescu_hurlin_fast(country_data, TREAT_COLS[0], OUT_COLS[0], 1)
per_call = (time.perf_counter() - t0) / 10
total_calls = (len(INJECTION_CELLS) * len(DELTAS) * N_SIM
               + N_PERM * len(GRID_CELLS) + N_NOISE * len(LAGS) + len(GRID_CELLS))
est_serial_min = total_calls * per_call / 60
print(f"\nDH timing: {per_call * 1000:.1f} ms/call → {total_calls:,} calls "
      f"≈ {est_serial_min:.1f} min serial (joblib cuts this several-fold)")
if est_serial_min > 30:
    N_SIM, N_PERM = 150, 100
    print(f"RUNTIME GUARD TRIGGERED — reduced to N_SIM={N_SIM}, N_PERM={N_PERM}")
else:
    print(f"Runtime OK — keeping N_SIM={N_SIM}, N_PERM={N_PERM}")

print("\n[Section 0] Setup complete — panel rebuilt and verified (192 countries).")

[checkpoint] loaded ← master_panel.parquet  (15,168 rows)
[rq2_panel] 6,912 rows, 192 countries, 1989–2024
Panel: (6912, 30), 192 countries, 1989–2024
Grid: 36 cells | injection cells: [('GROUND', 'part_n_minor'), ('MISSILES', 'part_n_minor')]



DH timing: 41.1 ms/call → 11,736 calls ≈ 8.0 min serial (joblib cuts this several-fold)
Runtime OK — keeping N_SIM=300, N_PERM=200

[Section 0] Setup complete — panel rebuilt and verified (192 countries).


### Section 0 — cache integrity fingerprint

Every cached simulation is only valid for the exact panel it was computed on. A
stable digest of the panel's treatment/outcome matrix lets each cache self-verify:
a fingerprint mismatch downgrades a cache to a MISS. Legacy caches with no
fingerprint column are accepted and stamped on the next fresh compute.

In [2]:
# --- Cache integrity: fingerprint the panel that feeds every cached simulation ---
import hashlib

# country column name (panel uses iso3; detect defensively)
ISO_COL = "iso3" if "iso3" in panel.columns else next(
    c for c in panel.columns if panel[c].dtype == object and c != "year")
if ISO_COL != "iso3":
    print(f"[Section 0] country column detected as '{ISO_COL}' (not 'iso3')")

FINGERPRINT_COLS = sorted(TREAT_COLS + OUT_COLS)

def panel_fingerprint(df, cols=FINGERPRINT_COLS):
    """Stable 16-hex digest of the panel's treat/outcome matrix.
    Requires df sorted by (iso3, year) so the byte order is deterministic."""
    d = df.sort_values([ISO_COL, "year"])[cols].to_numpy(dtype="float64")
    d = np.nan_to_num(d, nan=-9.87654321e30, posinf=1e300, neginf=-1e300)
    return hashlib.sha256(np.ascontiguousarray(d).tobytes()).hexdigest()[:16]

PANEL_FP = panel_fingerprint(panel)
print(f"[Section 0] Panel fingerprint: {PANEL_FP}  "
      f"({len(FINGERPRINT_COLS)} cols x {len(panel):,} rows)")

[Section 0] Panel fingerprint: 777a9d7cf8fefc3a  (7 cols x 6,912 rows)


## Section 1 — Power Curve: What CAN This Pipeline Detect?

Synthetic outcomes with a **known injected lead-lag effect**, pushed through the
real DH test:

`y_synth_it = y_real_it + δ · z(x_i,t−1)`

where `z()` standardizes the real treatment over the full panel, so δ is in
outcome-SD-per-treatment-SD units — comparable to a correlation. Using the *real*
outcome keeps the true noise structure, autocorrelation, and missingness. Each
simulation bootstrap-resamples countries with replacement (sampling variation),
injects, and runs DH at lag 1. Detection rate = share of sims with p < 0.05.

δ = 0 is the leftmost grid point — the **rejection-rate-at-δ=0 check**. But note
this is *not* a clean calibrated size measure: at δ=0 the synthetic outcome equals
the real outcome, so the rejection rate mixes over-rejection with any genuine
effect, and bootstrap resampling can inflate it. The calibrated size evidence is
Section 2b's permutation nulls, where the treatment→outcome link is destroyed.

In [3]:
# Pre-standardize each injection cell's treatment (pooled panel mean/sd) and
# precompute the lag-1 injection series per country
cell_data_by_cell = {}
for w, o in INJECTION_CELLS:
    tc, oc = f"d_log_tiv_{w}", f"d_log_{o}"
    mu, sd = np.nanmean(panel[tc]), np.nanstd(panel[tc])
    cdict = {}
    for iso3, arrs in country_data.items():
        zx = (arrs[tc] - mu) / sd
        cdict[iso3] = {"x": arrs[tc], "y": arrs[oc],
                       "zx_lag": np.concatenate(([np.nan], zx[:-1]))}
    cell_data_by_cell[f"{w}×{o}"] = cdict


def run_power_task(task_idx, cell_label, cell_dict, delta, n_sim, alpha, seed):
    """One (cell, δ) grid point: n_sim bootstrap+inject+DH simulations."""
    rng = np.random.default_rng([seed, task_idx])
    keys = list(cell_dict)
    rows = []
    for s in range(n_sim):
        sampled = rng.integers(0, len(keys), size=len(keys))
        boot = {}
        for j, ki in enumerate(sampled):
            d = cell_dict[keys[ki]]
            inj = d["zx_lag"]
            y = d["y"] + np.where(np.isfinite(inj), delta * inj, 0.0)
            boot[j] = {"x": d["x"], "y": y}
        Z, p, _ = dumitrescu_hurlin_fast(boot, "x", "y", 1)
        rows.append({"cell": cell_label, "delta": delta, "sim": s, "Z": Z, "p": p,
                     "detected": bool(np.isfinite(p) and (p < alpha))})
    return rows


POWER_CACHE = CACHE_DIR / "nb12_power_sims.parquet"
tasks = [(i, label, delta)
         for i, (label, delta) in enumerate(product(cell_data_by_cell, DELTAS))]
expected_rows = len(tasks) * N_SIM

cache_ok = False
if POWER_CACHE.exists():
    power_df = pd.read_parquet(POWER_CACHE)
    cache_ok = (len(power_df) == expected_rows
                and set(power_df["cell"]) == set(cell_data_by_cell)
                and len(set(power_df["delta"])) == len(DELTAS))
    if cache_ok:
        if "panel_fp" in power_df.columns:
            fp_ok = (power_df["panel_fp"].iloc[0] == PANEL_FP)
            if not fp_ok:
                print(f"  Cache fingerprint MISMATCH "
                      f"(cache={power_df['panel_fp'].iloc[0]}, live={PANEL_FP}) "
                      f"— treating as MISS")
        else:
            fp_ok = True
            print("  Legacy cache without panel_fp — fingerprint check SKIPPED "
                  "(will be stamped on next fresh compute)")
        cache_ok = cache_ok and fp_ok
if cache_ok:
    print(f"Cache HIT — loaded {len(power_df):,} sims from {POWER_CACHE.name}")
else:
    print(f"Cache MISS — running {len(tasks)} tasks × {N_SIM} sims ...")
    t0 = time.perf_counter()
    results = Parallel(n_jobs=-2)(
        delayed(run_power_task)(i, label, cell_data_by_cell[label], delta,
                                N_SIM, ALPHA, SEED)
        for i, label, delta in tasks)
    power_df = pd.DataFrame([r for rows in results for r in rows])
    power_df["panel_fp"] = PANEL_FP
    power_df.to_parquet(POWER_CACHE, index=False)
    print(f"Computed fresh in {time.perf_counter() - t0:.0f}s → cached to "
          f"{POWER_CACHE.name}")

power_curve = (power_df.groupby(["cell", "delta"])["detected"]
                       .mean().rename("power").reset_index())

# Rejection rate at δ = 0. Flag constants + banned-framing list live here so the
# δ=0 relabel can be VERIFIED by check [16] rather than trusted on attestation.
REJECT_ZERO_FLAG_HI = "HIGH REJECTION (not calibrated size)"
REJECT_ZERO_FLAG_LO = "rejection rate near nominal"
BANNED_SIZE_STRINGS = ["SIZE DISTORTION", "size distortion",
                       "Type-I size", "type-I size", "type-I error rate"]
reject_at_zero = {}
print("\n=== Rejection rate at δ=0 (real outcome, real treatment, bootstrap "
      "country resample) ===")
for cell in cell_data_by_cell:
    rate = power_curve.loc[(power_curve["cell"] == cell)
                           & (power_curve["delta"] == 0.0), "power"].iloc[0]
    reject_at_zero[cell] = rate
    flag = REJECT_ZERO_FLAG_HI if rate > 0.12 else REJECT_ZERO_FLAG_LO
    if rate > 0.12:
        print(f"  {cell}: {rate:.3f} — *** HIGH REJECTION AT δ=0 *** — at δ=0 "
              f"the outcome is unmodified, so this mixes over-rejection with any "
              f"genuine effect. See Section 2b for the permutation-based "
              f"calibrated size.")
    else:
        print(f"  {cell}: {rate:.3f} — {flag} at δ=0")

print("\nInterpretation caveat: δ=0 leaves the real outcome and real treatment "
      "in place, so this is a rejection rate under the observed data, not an "
      "error rate against a true null. Bootstrap resampling of countries with "
      "replacement "
      "duplicates units and can inflate it further. The calibrated size "
      "evidence is Section 2b's permutation nulls, where the treatment-outcome "
      "link is destroyed.")


reject_zero_ok = all(r <= 0.12 for r in reject_at_zero.values())


def power_crossing(deltas, powers, target=0.80):
    """First δ where power crosses target, linearly interpolated.
    0.0 if already above target at δ=0; NaN if never reached."""
    if powers and powers[0] >= target:
        return 0.0
    for i in range(1, len(deltas)):
        if powers[i - 1] < target <= powers[i]:
            d0, d1, p0, p1 = deltas[i - 1], deltas[i], powers[i - 1], powers[i]
            return d0 + (target - p0) * (d1 - d0) / (p1 - p0)
    return np.nan


delta_star = {}
print("\n=== Minimum detectable effect (80% power) ===")
for cell in cell_data_by_cell:
    sub = power_curve[power_curve["cell"] == cell].sort_values("delta")
    ds = power_crossing(sub["delta"].tolist(), sub["power"].tolist())
    delta_star[cell] = ds
    if np.isfinite(ds) and ds == 0.0:
        print(f"  {cell}: δ* ≈ 0 — rejection rate already ≥80% at δ=0; "
              f"not interpretable as power")
    elif np.isfinite(ds):
        print(f"  {cell}: δ* = {ds:.3f}")
    else:
        print(f"  {cell}: δ* > 0.30 (never reaches 80%)")

print("\nNOTE FOR THE MANUSCRIPT: do not report delta_star. A minimum "
      "detectable effect is not interpretable for a test whose rejection rate "
      "at delta=0 is far above nominal. Section 1 is a diagnostic, not a "
      "power claim.")

finite_ds = [d for d in delta_star.values() if np.isfinite(d)]
if finite_ds and all(MAX_OBSERVED_CCF < d for d in finite_ds):
    relation = "below"
elif finite_ds:
    relation = "at or above"
else:
    relation = "below"   # power never reaches 80% anywhere on the grid
caveat = ("" if reject_zero_ok else
          " Floor not interpretable: high rejection at δ=0 "
          "(see item 2b for calibrated size).")
print(f"\nThe pipeline detects standardized effects ≥ δ*; the largest observed raw "
      f"cross-correlation in NB06 was |ρ| = {MAX_OBSERVED_CCF} — {relation} the "
      f"detection floor.{caveat}")

power_out = power_curve.merge(
    pd.DataFrame([{"cell": c, "delta_star_80": round(d, 4) if np.isfinite(d) else np.nan}
                  for c, d in delta_star.items()]), on="cell")
power_out["n_sim"] = N_SIM
power_out.to_csv(TBL_DIR / "section1_power_curve.csv", index=False)

# Figure: power curve
fig, ax = plt.subplots(figsize=(8, 5))
colors = {list(cell_data_by_cell)[0]: "steelblue", list(cell_data_by_cell)[1]: "indianred"}
for cell in cell_data_by_cell:
    sub = power_curve[power_curve["cell"] == cell].sort_values("delta")
    ax.plot(sub["delta"], sub["power"], marker="o", lw=1.6,
            color=colors[cell], label=cell)
    if np.isfinite(delta_star[cell]):
        ax.axvline(delta_star[cell], color=colors[cell], ls=":", lw=1)
        ax.text(delta_star[cell], 0.35, f" δ*={delta_star[cell]:.2f}",
                rotation=90, fontsize=8, color=colors[cell], va="bottom")
ax.axhline(0.80, color="gray", ls="--", lw=0.9, label="80% power")
ax.axhline(ALPHA, color="gray", ls=":", lw=0.8)
ax.axvline(MAX_OBSERVED_CCF, color="black", ls="-.", lw=0.9,
           label=f"max observed |ρ| = {MAX_OBSERVED_CCF}")
ax.set_xlabel("injected effect δ (outcome SD per treatment SD)", fontsize=9)
ax.set_ylabel("detection rate (p < 0.05)", fontsize=9)
ax.set_ylim(0, 1.02)
ax.set_title("Power curve of the NB06 DH pipeline (lag 1)", fontsize=10)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig1_power_curve.png", dpi=300, bbox_inches="tight")
plt.close(fig)

print(f"\n[Section 1] Power curve done ({len(power_df):,} sims) — saved → "
      f"section1_power_curve.csv + fig1_power_curve.png")

  Legacy cache without panel_fp — fingerprint check SKIPPED (will be stamped on next fresh compute)
Cache HIT — loaded 4,200 sims from nb12_power_sims.parquet

=== Rejection rate at δ=0 (real outcome, real treatment, bootstrap country resample) ===
  GROUND×part_n_minor: 0.540 — *** HIGH REJECTION AT δ=0 *** — at δ=0 the outcome is unmodified, so this mixes over-rejection with any genuine effect. See Section 2b for the permutation-based calibrated size.
  MISSILES×part_n_minor: 0.880 — *** HIGH REJECTION AT δ=0 *** — at δ=0 the outcome is unmodified, so this mixes over-rejection with any genuine effect. See Section 2b for the permutation-based calibrated size.

Interpretation caveat: δ=0 leaves the real outcome and real treatment in place, so this is a rejection rate under the observed data, not an error rate against a true null. Bootstrap resampling of countries with replacement duplicates units and can inflate it further. The calibrated size evidence is Section 2b's permutation nulls


[Section 1] Power curve done (4,200 sims) — saved → section1_power_curve.csv + fig1_power_curve.png


## Section 2 — Permutation Null: Is 8 Significant Cells More Than Chance?

Each permutation shuffles every treatment column **within country** (NB06's
placebo pattern, preserving each country's outcome series and treatment marginal
distribution), then runs the full 36-cell DH grid and counts cells with p < 0.05.
The observed count (8) is compared against this null distribution;
empirical one-sided p = (1 + #{null ≥ 8}) / (1 + N_PERM). Per-cell Z values are
stored for Section 3's artifact diagnosis.

In [4]:
def run_perm_task(perm_idx, cdata, treat_cols, grid_cells, seed):
    """One permutation: within-country shuffle of all treatments, full DH grid."""
    rng = np.random.default_rng([seed, 100_000 + perm_idx])
    shuffled = {}
    for iso3, arrs in cdata.items():
        new = dict(arrs)
        for tc in treat_cols:
            x = arrs[tc].copy()
            valid = np.isfinite(x)
            if valid.sum() > 1:
                x[valid] = rng.permutation(x[valid])
            new[tc] = x
        shuffled[iso3] = new
    out = []
    for (w, o, lag) in grid_cells:
        Z, p, _ = dumitrescu_hurlin_fast(shuffled, f"d_log_tiv_{w}", f"d_log_{o}", lag)
        out.append({"perm": perm_idx, "weapon": w, "outcome": o, "lag": lag,
                    "Z": Z, "p": p})
    return out


PERM_CACHE = CACHE_DIR / "nb12_perm_counts.parquet"
expected_rows = N_PERM * len(GRID_CELLS)

cache_ok = False
if PERM_CACHE.exists():
    perm_df = pd.read_parquet(PERM_CACHE)
    cache_ok = (len(perm_df) == expected_rows
                and perm_df["perm"].nunique() == N_PERM)
    if cache_ok:
        if "panel_fp" in perm_df.columns:
            fp_ok = (perm_df["panel_fp"].iloc[0] == PANEL_FP)
            if not fp_ok:
                print(f"  Cache fingerprint MISMATCH "
                      f"(cache={perm_df['panel_fp'].iloc[0]}, live={PANEL_FP}) "
                      f"— treating as MISS")
        else:
            fp_ok = True
            print("  Legacy cache without panel_fp — fingerprint check SKIPPED "
                  "(will be stamped on next fresh compute)")
        cache_ok = cache_ok and fp_ok
if cache_ok:
    print(f"Cache HIT — loaded {len(perm_df):,} cell results from {PERM_CACHE.name}")
else:
    print(f"Cache MISS — running {N_PERM} permutations × {len(GRID_CELLS)} cells ...")
    t0 = time.perf_counter()
    results = Parallel(n_jobs=-2)(
        delayed(run_perm_task)(i, country_data, TREAT_COLS, GRID_CELLS, SEED)
        for i in range(N_PERM))
    perm_df = pd.DataFrame([r for rows in results for r in rows])
    perm_df["panel_fp"] = PANEL_FP
    perm_df.to_parquet(PERM_CACHE, index=False)
    print(f"Computed fresh in {time.perf_counter() - t0:.0f}s → cached to "
          f"{PERM_CACHE.name}")

perm_counts = (perm_df.assign(sig=perm_df["p"] < ALPHA)
                      .groupby("perm")["sig"].sum())
null_mean, null_sd = perm_counts.mean(), perm_counts.std()
emp_p = (1 + (perm_counts >= OBSERVED_SIG_CELLS).sum()) / (1 + N_PERM)

print(f"\nNull significant-cell count: mean={null_mean:.2f}, sd={null_sd:.2f}, "
      f"range=[{perm_counts.min()}, {perm_counts.max()}]")
print(f"Observed (NB06): {OBSERVED_SIG_CELLS}")
print(f"Empirical one-sided p = (1 + {(perm_counts >= OBSERVED_SIG_CELLS).sum()}) "
      f"/ (1 + {N_PERM}) = {emp_p:.4f}")
if emp_p >= 0.10:
    perm_reading = ("8/36 nominal significances are indistinguishable from chance "
                    "under within-country permutation")
else:
    perm_reading = ("the count exceeds chance — the null rests on the placebo "
                    "rates, report both")
print(f"Reading: {perm_reading}")

counts_out = perm_counts.reset_index().rename(columns={"sig": "n_sig_cells"})
counts_out["perm"] = counts_out["perm"].astype(str)
summary = pd.DataFrame([
    {"perm": "OBSERVED_NB06", "n_sig_cells": OBSERVED_SIG_CELLS},
    {"perm": "NULL_MEAN",     "n_sig_cells": round(null_mean, 3)},
    {"perm": "NULL_SD",       "n_sig_cells": round(null_sd, 3)},
    {"perm": "EMPIRICAL_P",   "n_sig_cells": round(emp_p, 4)},
])
pd.concat([counts_out, summary], ignore_index=True).to_csv(
    TBL_DIR / "section2_perm_null.csv", index=False)

# Figure: null histogram with observed count marked
fig, ax = plt.subplots(figsize=(8, 4.5))
bins = np.arange(-0.5, max(perm_counts.max(), OBSERVED_SIG_CELLS) + 1.5, 1)
ax.hist(perm_counts, bins=bins, color="steelblue", edgecolor="white")
ax.axvline(OBSERVED_SIG_CELLS, color="indianred", lw=1.6,
           label=f"observed = {OBSERVED_SIG_CELLS} (empirical p={emp_p:.3f})")
ax.set_xlabel("significant cells (p < 0.05) out of 36, per permutation", fontsize=9)
ax.set_ylabel("permutations", fontsize=9)
ax.set_title("Permutation null of the significant-cell count", fontsize=10)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig2_permutation_null.png", dpi=300, bbox_inches="tight")
plt.close(fig)

print(f"\n[Section 2] Permutation null done ({N_PERM} permutations) — saved → "
      f"section2_perm_null.csv + fig2_permutation_null.png")

  Legacy cache without panel_fp — fingerprint check SKIPPED (will be stamped on next fresh compute)
Cache HIT — loaded 7,200 cell results from nb12_perm_counts.parquet

Null significant-cell count: mean=3.80, sd=1.88, range=[0, 8]
Observed (NB06): 8
Empirical one-sided p = (1 + 5) / (1 + 200) = 0.0299
Reading: the count exceeds chance — the null rests on the placebo rates, report both



[Section 2] Permutation null done (200 permutations) — saved → section2_perm_null.csv + fig2_permutation_null.png


## Section 2b — Per-Cell Permutation Reconciliation (hardened)

Section 2 compared the **count** of nominally significant cells (8) against its
permutation null. But the 36 cells are not independent draws — they share
countries, three outcome series, and four treatment series — so the count is not
binomial and its null is not a clean reference distribution. The sharper test is
**per-cell**: compare each observed lag-1 Z against that same cell's own
permutation Z distribution.

This hardened version fixes four issues in the first pass:

1. **One-sided p_cell.** `dumitrescu_hurlin_fast` returns a one-sided upper-tail p
   (`p = 1 − Φ(Z)`), so `nominal_sig` and the per-cell filter must both use the
   upper tail. The first pass used a two-sided `|Z|` comparison, mismatching the
   tails and making the filter mechanically conservative.
2. **Finite denominator.** `p_cell` divides by the number of finite permutation
   draws, not by `N_PERM`.
3. **2000-permutation lag-1 null.** A dedicated high-resolution cache
   (disjoint seed stream) drops the Monte-Carlo SE near p≈0.05 from ~0.015 to
   ~0.005, so threshold decisions are resolvable.
4. **Multiplicity.** Benjamini-Hochberg and Bonferroni across the pre-specified
   12-cell lag-1 family, because 12 tests at α=0.05 expect 0.6 false positives.

The permutation nulls also double as the **calibrated size evidence**: with the
treatment→outcome link destroyed, DH's Z̄̃ should be N(0,1); departures quantify
over-rejection directly, covering every lag-1 cell.

In [5]:
# --- What is the third return value of dumitrescu_hurlin_fast? ---
_z, _p, _aux = dumitrescu_hurlin_fast(country_data,
                                      "d_log_tiv_GROUND",
                                      "d_log_part_n_war", 1)
print(f"Z={_z:.4f}  p={_p:.4f}")
print(f"aux type: {type(_aux)}")
if isinstance(_aux, dict):
    print(f"aux keys: {list(_aux)[:12]}")
    for k in list(_aux)[:3]:
        print(f"  {k}: {type(_aux[k])} -> {str(_aux[k])[:200]}")
elif hasattr(_aux, "__len__"):
    print(f"aux len={len(_aux)}  head={str(_aux[:8])[:300]}")
else:
    print(f"aux value: {_aux}")

MIN_OBS_LAG1 = 5  # minimum finite paired observations for a lag-1 per-country fit

def contributing_units(cdata, tcol, ocol, lag=1, min_obs=MIN_OBS_LAG1):
    n = 0
    for arrs in cdata.values():
        x, y = arrs[tcol], arrs[ocol]
        m = np.isfinite(x) & np.isfinite(y)
        if m.sum() >= min_obs + lag:
            n += 1
    return n

print(f"\naux is a SCALAR (the DH test's own contributing-unit count, filtered "
      f"at its internal min_obs=12) — it exposes neither per-country Wald stats "
      f"nor a lag-1-specific count. Falling back to contributing_units() at "
      f"min_obs={MIN_OBS_LAG1} for the n_units diagnostic column.")
print(f"  example GROUND×part_n_war: aux={_aux}, "
      f"contributing_units={contributing_units(country_data, 'd_log_tiv_GROUND', 'd_log_part_n_war', 1)}")
print("[Section 2b] DH auxiliary inspected; contributing_units() defined.")

Z=6.4233  p=0.0000
aux type: <class 'int'>
aux value: 123

aux is a SCALAR (the DH test's own contributing-unit count, filtered at its internal min_obs=12) — it exposes neither per-country Wald stats nor a lag-1-specific count. Falling back to contributing_units() at min_obs=5 for the n_units diagnostic column.
  example GROUND×part_n_war: aux=123, contributing_units=192
[Section 2b] DH auxiliary inspected; contributing_units() defined.


In [6]:
# ================= Section 2b — high-resolution lag-1 permutation null =========
N_PERM_2B = 2000
LAG1_CELLS = [(w, o) for w in WEAPON_CLASS_NAMES for o in OUTCOMES]
LAG1_GRID  = [(w, o, 1) for (w, o) in LAG1_CELLS]

def run_perm_task_lag1(perm_idx, cdata, treat_cols, grid_cells, seed):
    """One permutation, lag-1 cells only. Within-country shuffle of every
    treatment column, preserving each country's outcome series and each
    treatment's marginal distribution. Mirrors run_perm_task exactly except
    for the restricted grid and the disjoint seed stream."""
    rng = np.random.default_rng([seed, 500_000 + perm_idx])
    shuffled = {}
    for iso3, arrs in cdata.items():
        new = dict(arrs)
        for tc in treat_cols:
            x = arrs[tc].copy()
            valid = np.isfinite(x)
            if valid.sum() > 1:
                x[valid] = rng.permutation(x[valid])
            new[tc] = x
        shuffled[iso3] = new
    out = []
    for (w, o, lag) in grid_cells:
        Z, p, _ = dumitrescu_hurlin_fast(shuffled, f"d_log_tiv_{w}",
                                         f"d_log_{o}", lag)
        out.append({"perm": perm_idx, "weapon": w, "outcome": o, "lag": lag,
                    "Z": Z, "p": p})
    return out

PERM2B_CACHE = CACHE_DIR / f"nb12_perm_lag1_{N_PERM_2B}.parquet"
expected_rows_2b = N_PERM_2B * len(LAG1_GRID)

cache_ok_2b = False
if PERM2B_CACHE.exists():
    perm2b_df = pd.read_parquet(PERM2B_CACHE)
    cache_ok_2b = (len(perm2b_df) == expected_rows_2b
                   and perm2b_df["perm"].nunique() == N_PERM_2B
                   and set(zip(perm2b_df["weapon"], perm2b_df["outcome"]))
                       == set(LAG1_CELLS))
    if cache_ok_2b and "panel_fp" in perm2b_df.columns:
        if perm2b_df["panel_fp"].iloc[0] != PANEL_FP:
            print(f"  2b cache fingerprint MISMATCH — treating as MISS")
            cache_ok_2b = False

if cache_ok_2b:
    print(f"Cache HIT — loaded {len(perm2b_df):,} lag-1 cell results "
          f"from {PERM2B_CACHE.name}")
else:
    est_calls = N_PERM_2B * len(LAG1_GRID)
    print(f"Cache MISS — running {N_PERM_2B:,} permutations x "
          f"{len(LAG1_GRID)} lag-1 cells = {est_calls:,} DH calls "
          f"(~{est_calls * 0.0241 / 60:.1f} min serial, less under joblib)")
    t0 = time.perf_counter()
    results_2b = Parallel(n_jobs=-2)(
        delayed(run_perm_task_lag1)(i, country_data, TREAT_COLS,
                                    LAG1_GRID, SEED)
        for i in range(N_PERM_2B))
    perm2b_df = pd.DataFrame([r for rows in results_2b for r in rows])
    perm2b_df["panel_fp"] = PANEL_FP
    perm2b_df.to_parquet(PERM2B_CACHE, index=False)
    print(f"Computed fresh in {time.perf_counter() - t0:.0f}s "
          f"-> cached to {PERM2B_CACHE.name}")

print(f"[Section 2b] Permutation basis: N_PERM_2B={N_PERM_2B}, "
      f"{len(LAG1_CELLS)} lag-1 cells, seed stream 500000+i "
      f"(disjoint from Section 2's 100000+i)")

Cache HIT — loaded 24,000 lag-1 cell results from nb12_perm_lag1_2000.parquet
[Section 2b] Permutation basis: N_PERM_2B=2000, 12 lag-1 cells, seed stream 500000+i (disjoint from Section 2's 100000+i)


In [7]:
from statsmodels.stats.multitest import multipletests

sec2b_rows = []
for w, o in LAG1_CELLS:
    tcol, ocol = f"d_log_tiv_{w}", f"d_log_{o}"
    obs_Z, obs_p, obs_aux = dumitrescu_hurlin_fast(country_data, tcol, ocol, 1)

    null_all = perm2b_df[(perm2b_df["weapon"] == w)
                         & (perm2b_df["outcome"] == o)
                         & (perm2b_df["lag"] == 1)]["Z"].to_numpy()
    null_finite = null_all[np.isfinite(null_all)]
    n_finite = int(null_finite.size)

    # ONE-SIDED upper tail, matching the DH Z-bar-tilde convention that
    # produces obs_p. Denominator is the finite draw count, not N_PERM_2B.
    n_exceed = int(np.sum(null_finite >= obs_Z))
    p_cell = (1 + n_exceed) / (1 + n_finite) if n_finite > 0 else np.nan

    null_mean_Z = float(np.mean(null_finite)) if n_finite else np.nan
    null_sd_Z   = float(np.std(null_finite))  if n_finite else np.nan
    z_pos = ((obs_Z - null_mean_Z) / null_sd_Z
             if (n_finite and null_sd_Z > 0) else np.nan)

    sec2b_rows.append({
        "weapon": w, "outcome": o,
        "obs_Z": round(obs_Z, 4), "obs_p": round(obs_p, 4),
        "null_mean_Z": round(null_mean_Z, 4),
        "null_sd_Z": round(null_sd_Z, 4),
        "z_pos": round(z_pos, 4),
        "n_exceed": n_exceed,
        "n_finite_perm": n_finite,
        "n_nonfinite_perm": int(N_PERM_2B - n_finite),
        "n_units_dh": int(obs_aux),      # DH's own count, internal min_obs=12
        "n_units_loose": contributing_units(country_data, tcol, ocol, 1),
        "p_cell": round(p_cell, 5),
        "nominal_sig": bool(obs_p < ALPHA),
    })

sec2b_df = pd.DataFrame(sec2b_rows)

# --- Multiplicity across the pre-specified 12-cell lag-1 family -------------
_bh   = multipletests(sec2b_df["p_cell"].values, alpha=ALPHA, method="fdr_bh")
_bonf = multipletests(sec2b_df["p_cell"].values, alpha=ALPHA, method="bonferroni")
sec2b_df["q_bh"]        = np.round(_bh[1], 4)
sec2b_df["p_bonf"]      = np.round(_bonf[1], 4)
sec2b_df["survives_percell"]     = sec2b_df["p_cell"] < ALPHA
sec2b_df["survives_bh"]          = _bh[0]
sec2b_df["survives_bonferroni"]  = _bonf[0]

sec2b_df = sec2b_df.sort_values("p_cell").reset_index(drop=True)

print("=== Section 2b — per-cell permutation test "
      f"(lag 1, {len(LAG1_CELLS)} cells, {N_PERM_2B:,} permutations, "
      f"ONE-SIDED upper tail) ===\n")
_show = ["weapon", "outcome", "obs_Z", "obs_p", "null_mean_Z", "null_sd_Z",
         "z_pos", "p_cell", "q_bh", "p_bonf", "n_units_dh", "n_finite_perm",
         "nominal_sig", "survives_percell", "survives_bh"]
print(sec2b_df[_show].to_string(index=False))

n_nominal_sig      = int(sec2b_df["nominal_sig"].sum())
n_percell_survivors = int((sec2b_df["nominal_sig"]
                           & sec2b_df["survives_percell"]).sum())
n_all_survivors     = int(sec2b_df["survives_percell"].sum())
n_bh_survivors      = int(sec2b_df["survives_bh"].sum())
n_bonf_survivors    = int(sec2b_df["survives_bonferroni"].sum())

# per-cell permutation rejection rate (computed once; reused below and by size_ok)
perm_size_rates = (perm2b_df[perm2b_df["lag"] == 1]
                   .groupby(["weapon", "outcome"])["p"]
                   .apply(lambda s: float(np.mean(s.dropna() < ALPHA))))

# --- Does the null's dispersion track the DH contributing-unit count? -------
print("\n=== Null dispersion vs DH contributing units ===")
print("contributing_units() at min_obs=5 is uninformative (all 192); the DH "
      "test's own count at its internal min_obs=12 is the number that enters "
      "the statistic.")
_disp = sec2b_df[["weapon", "outcome", "n_units_dh", "n_units_loose",
                  "null_sd_Z", "null_mean_Z"]].sort_values("null_sd_Z",
                                                           ascending=False)
print(_disp.to_string(index=False))

_r_sd, _p_sd = stats.spearmanr(sec2b_df["n_units_dh"], sec2b_df["null_sd_Z"])
_r_mu, _p_mu = stats.spearmanr(sec2b_df["n_units_dh"], sec2b_df["null_mean_Z"])
print(f"\nSpearman rho(n_units_dh, null_sd_Z)   = {_r_sd:+.3f} (p={_p_sd:.4f})")
print(f"Spearman rho(n_units_dh, null_mean_Z) = {_r_mu:+.3f} (p={_p_mu:.4f})")
print(f"n_units_dh range: {sec2b_df['n_units_dh'].min()} to "
      f"{sec2b_df['n_units_dh'].max()} "
      f"(n_units_loose is constant at {sec2b_df['n_units_loose'].iloc[0]})")

_hi = sec2b_df.nlargest(2, "null_sd_Z")
_lo = sec2b_df.nsmallest(2, "null_sd_Z")
print(f"\nHighest-dispersion cells: "
      + "; ".join(f"{r.weapon}x{r.outcome} sd={r.null_sd_Z:.2f} "
                  f"N_dh={r.n_units_dh}" for r in _hi.itertuples()))
print(f"Lowest-dispersion cells:  "
      + "; ".join(f"{r.weapon}x{r.outcome} sd={r.null_sd_Z:.2f} "
                  f"N_dh={r.n_units_dh}" for r in _lo.itertuples()))
if _r_sd < -0.4 and _p_sd < 0.10:
    disp_reading = ("Dispersion is driven by thin cross-sectional support: "
                    "cells with fewer countries clearing DH's min_obs have "
                    "markedly heavier-tailed nulls. The heavy tails have a "
                    "mechanism, not just a description.")
elif _r_sd > 0.4 and _p_sd < 0.10:
    disp_reading = ("Dispersion RISES with contributing units, which is the "
                    "opposite of a thin-support explanation. Report as an "
                    "open feature of DH on this panel; do not attribute the "
                    "heavy tails to sparse cells.")
else:
    disp_reading = (f"No monotone relation between contributing units and "
                    f"null dispersion (rho={_r_sd:+.3f}, p={_p_sd:.4f}) across "
                    f"only {len(sec2b_df)} cells. The heavy tails are real "
                    f"(zero non-finite draws, and check [15] cross-validates "
                    f"them against an independent seed stream) but this "
                    f"notebook does not isolate their source. State that "
                    f"plainly rather than gesturing at sparsity.")
print(f"\nReading: {disp_reading}")

# --- Outcome degeneracy: the variable n_units_dh cannot see -----------------
DEGEN_VAR_TOL = 1e-10

def degeneracy_profile(cdata, tcol, ocol, lag=1, min_obs=12):
    n_elig = n_zero_var = 0
    ratios = []
    for arrs in cdata.values():
        x, y = arrs[tcol], arrs[ocol]
        m = np.isfinite(x) & np.isfinite(y)
        if m.sum() < min_obs + lag:
            continue
        n_elig += 1
        yv = y[m]
        if np.var(yv) <= DEGEN_VAR_TOL:
            n_zero_var += 1
            continue
        xl, yy = x[m][:-lag], yv[lag:]
        if np.var(xl) <= DEGEN_VAR_TOL:
            continue
        r = np.corrcoef(xl, yy)[0, 1]
        if np.isfinite(r) and abs(r) < 1.0:
            ratios.append(r ** 2 / max(1.0 - r ** 2, 1e-12))
    return {"n_eligible": n_elig, "n_zero_var_outcome": n_zero_var,
            "frac_zero_var": (n_zero_var / n_elig) if n_elig else np.nan,
            "wald_p99": float(np.percentile(ratios, 99)) if ratios else np.nan,
            "wald_max": float(np.max(ratios)) if ratios else np.nan}

_dg = pd.DataFrame([
    {"weapon": w, "outcome": o,
     **degeneracy_profile(country_data, f"d_log_tiv_{w}", f"d_log_{o}", 1)}
    for w, o in LAG1_CELLS])
_dg = _dg.merge(sec2b_df[["weapon", "outcome", "null_sd_Z", "n_units_dh"]],
                on=["weapon", "outcome"]).sort_values("null_sd_Z",
                                                      ascending=False)
print("\n  Methods note: DH's internal min_obs filter counts observations, "
      "not variation, so a country contributing twelve zeros clears the "
      "filter and then produces an exploding Wald statistic. This is a "
      "general caveat for applying DH to sparse count outcomes, independent "
      "of this paper's findings.")
print("\n=== Outcome degeneracy vs null dispersion ===")
print(_dg.to_string(index=False))
for _v in ["frac_zero_var", "wald_p99", "n_units_dh"]:
    _r, _p = stats.spearmanr(_dg[_v], _dg["null_sd_Z"], nan_policy="omit")
    print(f"  Spearman rho({_v:<18s}, null_sd_Z) = {_r:+.3f} (p={_p:.4f})")
_r_dg, _p_dg = stats.spearmanr(_dg["frac_zero_var"], _dg["null_sd_Z"],
                               nan_policy="omit")
if _r_dg > 0.5 and _p_dg < 0.10:
    disp_reading_2 = (
        f"Dispersion tracks OUTCOME DEGENERACY, not unit counts "
        f"(rho={_r_dg:+.3f}, p={_p_dg:.4f}). Countries whose differenced "
        f"outcome is near-constant still clear DH's min_obs, contribute "
        f"near-degenerate per-country regressions, and produce exploding Wald "
        f"statistics. That is the mechanism behind the heavy tails, and it "
        f"explains why n_units_dh showed nothing.")
else:
    disp_reading_2 = (
        f"Outcome degeneracy does not explain the dispersion either "
        f"(rho={_r_dg:+.3f}, p={_p_dg:.4f}) across {len(_dg)} cells. Both "
        f"candidate mechanisms are now tested and neither isolates the "
        f"source. The heavy tails are real (zero non-finite draws, "
        f"cross-validated against an independent seed stream in check [15]) "
        f"but unexplained. State that.")
print(f"\n  Reading: {disp_reading_2}")

_bonf_n_predictors = 3  # frac_zero_var, wald_p99, n_units_dh tested above
_p_dg_bonf = min(1.0, _p_dg * _bonf_n_predictors)
_gw_row = _dg[(_dg["weapon"] == "GROUND")
             & (_dg["outcome"] == "part_n_war")].iloc[0]
print(f"\n  Caveat: rho={_r_dg:+.3f} (p={_p_dg:.4f}) was one of "
      f"{_bonf_n_predictors} predictors tested; Bonferroni over "
      f"{_bonf_n_predictors} gives p={_p_dg_bonf:.4f}. The rank correlation "
      f"alone is suggestive, not established. What carries the finding is "
      f"the direct evidence: {int(_gw_row['n_zero_var_outcome'])} of "
      f"{int(_gw_row['n_eligible'])} countries "
      f"({_gw_row['frac_zero_var'] * 100:.0f}%) with near-constant "
      f"differenced outcomes in GROUND×part_n_war, and "
      f"wald_max={_gw_row['wald_max']:.1f}. Report the direct numbers as "
      f"primary, the rho as supporting.")

_dg.to_csv(TBL_DIR / "section2b_degeneracy.csv", index=False)

# --- Matched count test, lag 1 only, at 2000-permutation resolution ---------
# The 36-cell count includes 24 lag-2/3 cells whose Z is strongly negative and
# which therefore essentially cannot reject a one-sided upper-tail test. The
# comparable question is 8 of 12 at lag 1. Basis is perm2b_df (2000 perms),
# not perm_df (200), so the threshold decision is resolvable.
_p1_2b = perm2b_df[perm2b_df["lag"] == 1]
null_counts_lag1 = (_p1_2b.assign(sig=_p1_2b["p"] < ALPHA)
                          .groupby("perm")["sig"].sum())
obs_count_lag1 = int(sec2b_df["nominal_sig"].sum())
n_exceed_count = int((null_counts_lag1 >= obs_count_lag1).sum())
emp_p_lag1 = (1 + n_exceed_count) / (1 + len(null_counts_lag1))
mc_se_count = np.sqrt(emp_p_lag1 * (1 - emp_p_lag1) / len(null_counts_lag1))

print(f"\n[Section 2 supplement] Lag-1-only count test "
      f"({len(LAG1_CELLS)} cells, {N_PERM_2B:,} permutations):")
print(f"  null mean={null_counts_lag1.mean():.3f} "
      f"(sd={null_counts_lag1.std():.3f}), observed={obs_count_lag1}, "
      f"exceedances={n_exceed_count}/{len(null_counts_lag1)}, "
      f"empirical p={emp_p_lag1:.4f} (MC SE {mc_se_count:.4f})")

_p1_old = perm_df[perm_df["lag"] == 1]
_nc_old = (_p1_old.assign(sig=_p1_old["p"] < ALPHA)
                  .groupby("perm")["sig"].sum())
_emp_p_lag1_old = ((1 + int((_nc_old >= obs_count_lag1).sum()))
                   / (1 + len(_nc_old)))
print(f"  for reference: 200-perm basis gave p={_emp_p_lag1_old:.4f}; "
      f"36-cell 200-perm version gave p={emp_p:.4f} "
      f"(24 of those 36 cells are lag 2-3 and structurally non-rejecting)")

_mean_rate = float(null_counts_lag1.mean()) / len(LAG1_CELLS)
print(f"\n  Consistency: null count {null_counts_lag1.mean():.2f}/"
      f"{len(LAG1_CELLS)} = {_mean_rate:.3f} mean rejection rate, against "
      f"per-cell permutation rates {perm_size_rates.min():.3f}-"
      f"{perm_size_rates.max():.3f} (median {perm_size_rates.median():.3f}). "
      f"The count null is the over-rejection measured a second way.")

# ===== Cross-cell dependence: MEASUREMENTS ONLY (inference retracted below) ====
# These measure how correlated the 12 cells' permutation Z are, and whether the
# within-country shuffle changes cross-treatment correlation. The previous
# revision used them to argue the count null was anti-conservative; that
# inference is RETRACTED in the Conclusion below. The measurements are kept.
print("\n" + "=" * 74)
print("Cross-cell dependence — measurements (inference retracted below)")
print("=" * 74)

# Evidence 1 — the 12 cells' permutation Z are strongly correlated with
# each other, so the 12 tests are nowhere near 12 independent tests.
_zmat = (_p1_2b.assign(cell=_p1_2b["weapon"] + "x" + _p1_2b["outcome"])
               .pivot(index="perm", columns="cell", values="Z"))
_zcorr = _zmat.corr()
_off = _zcorr.where(~np.eye(len(_zcorr), dtype=bool)).stack()
print(f"\nEvidence 1 — correlation of lag-1 Z across the {N_PERM_2B:,} "
      f"permutations, {len(_zcorr)} cells:")
print(f"  off-diagonal correlation: min={_off.min():+.3f}, "
      f"median={_off.median():+.3f}, max={_off.max():+.3f}, "
      f"mean|r|={_off.abs().mean():.3f}")

# Same-outcome and same-weapon pairs isolated
_pairs = []
for a, b in _off.index:
    wa, oa = a.split("x", 1)
    wb, ob = b.split("x", 1)
    kind = ("same outcome" if oa == ob else
            "same weapon" if wa == wb else "neither shared")
    _pairs.append({"kind": kind, "r": _off.loc[(a, b)]})
_pairs = pd.DataFrame(_pairs)
print("  by shared series:")
for kind, g in _pairs.groupby("kind"):
    print(f"    {kind:<16s} n={len(g):>3d}  mean r={g['r'].mean():+.3f}  "
          f"mean|r|={g['r'].abs().mean():.3f}")

# Effective number of independent tests (Cheverud/Nyholt eigenvalue estimator)
_eig = np.linalg.eigvalsh(_zcorr.values)
_M = len(_zcorr)
_M_eff = 1 + (_M - 1) * (1 - np.var(_eig, ddof=1) / _M)
print(f"  effective independent tests (eigenvalue estimator): "
      f"M_eff = {_M_eff:.2f} of M = {_M}")

# Evidence 2 — the permutation destroys treatment-treatment correlation
_tcorr_obs = panel[TREAT_COLS].corr()
_obs_off = _tcorr_obs.where(~np.eye(len(TREAT_COLS), dtype=bool)).stack()
_N_SHUF_CHECK = 50
_shuf_means = []
_rng_chk = np.random.default_rng([SEED, 900_000])
for _s in range(_N_SHUF_CHECK):
    _tmp = panel[TREAT_COLS].copy()
    for _tc in TREAT_COLS:
        _tmp[_tc] = (panel.groupby(ISO_COL)[_tc]
                          .transform(lambda v: _rng_chk.permutation(v.values)))
    _c = _tmp.corr()
    _shuf_means.append(
        _c.where(~np.eye(len(TREAT_COLS), dtype=bool)).stack().abs().mean())
print(f"\nEvidence 2 — treatment-treatment correlation, observed vs permuted:")
print(f"  observed mean|r| across {len(TREAT_COLS)} treatment columns: "
      f"{_obs_off.abs().mean():.3f} "
      f"(range {_obs_off.min():+.3f} to {_obs_off.max():+.3f})")
print(f"  after within-country independent shuffling ({_N_SHUF_CHECK} draws): "
      f"mean|r| = {np.mean(_shuf_means):.3f} "
      f"(sd {np.std(_shuf_means):.4f})")
# The measurements stand; the inference drawn from them in the previous
# revision does not. Recorded here as a NEGATIVE result.
_obs_tr = float(_obs_off.abs().mean())
_perm_tr = float(np.mean(_shuf_means))
print(f"\nConclusion: BOTH legs of the cross-cell-dependence explanation FAIL.")
print(f"  Leg 1 — the 12 cells are near-INDEPENDENT: mean|r|="
      f"{_off.abs().mean():.3f}, median r={_off.median():+.3f}, "
      f"M_eff={_M_eff:.2f} of {len(_zcorr)}. The same-weapon subset "
      f"(mean|r|={_pairs.loc[_pairs['kind']=='same weapon','r'].abs().mean():.3f}"
      f", n={int((_pairs['kind']=='same weapon').sum())}) is real but confined "
      f"to 3-cell blocks and cannot inflate a 12-cell count null twofold.")
print(f"  Leg 2 — there was no cross-treatment correlation to destroy: "
      f"observed mean|r|={_obs_tr:.3f} is negligible in ABSOLUTE terms, "
      f"whatever its ratio to the permuted {_perm_tr:.3f}. A ratio test "
      f"against a near-zero baseline is uninformative and the earlier "
      f"guard on this comparison was invalid.")

count_dependence_reading = (
    f"NEGATIVE RESULT — the count excess (p={emp_p_lag1:.4f}) is NOT explained "
    f"by cross-cell dependence. The 12 cells are near-independent "
    f"(M_eff={_M_eff:.2f}/12) and observed cross-treatment correlation is "
    f"negligible (mean|r|={_obs_tr:.3f}). This revision retracts the "
    f"dependence explanation asserted in the previous revision. The aggregate "
    f"excess is characterised in Section 2e (omnibus tests) and its likely "
    f"source tested in Section 2f (circular-shift null).")
print(f"\n{count_dependence_reading}")

pd.DataFrame({"metric": ["obs_count_lag1", "null_mean", "null_sd", "emp_p_lag1",
                         "mc_se", "n_perm", "mean_abs_r_cellZ", "M_eff",
                         "obs_treat_mean_abs_r", "perm_treat_mean_abs_r"],
              "value": [obs_count_lag1, null_counts_lag1.mean(),
                        null_counts_lag1.std(), emp_p_lag1, mc_se_count,
                        N_PERM_2B, _off.abs().mean(), _M_eff,
                        _obs_off.abs().mean(), np.mean(_shuf_means)]}
            ).to_csv(TBL_DIR / "section2b_count_dependence.csv", index=False)
pd.DataFrame([{"metric": "dependence_explains_count", "value": 0}]).to_csv(
    TBL_DIR / "section2b_count_dependence.csv", mode="a", header=False, index=False)
_zcorr.to_csv(TBL_DIR / "section2b_cellZ_correlation.csv")

# --- Monte Carlo resolution at the threshold -------------------------------
mc_se = np.sqrt(ALPHA * (1 - ALPHA) / N_PERM_2B)
print(f"\nMonte Carlo SE on p_cell near {ALPHA}: {mc_se:.4f} "
      f"({N_PERM_2B:,} permutations). Cells within 2 SE of the threshold "
      f"are not resolvable:")
_border = sec2b_df[(sec2b_df["p_cell"] - ALPHA).abs() <= 2 * mc_se]
if len(_border):
    for r in _border.itertuples():
        print(f"  BORDERLINE {r.weapon}x{r.outcome}: p_cell={r.p_cell:.4f}")
else:
    print("  none")

# --- Permutation nulls as the primary size evidence ------------------------
print(f"\n=== Size evidence from the permutation nulls "
      f"(all {len(LAG1_CELLS)} lag-1 cells) ===")
print("Under permutation the treatment->outcome link is destroyed, so DH's "
      "Z-bar-tilde should be N(0,1).")
print(f"  observed null mean Z: {sec2b_df['null_mean_Z'].min():+.3f} to "
      f"{sec2b_df['null_mean_Z'].max():+.3f}   (nominal 0)")
print(f"  observed null sd   Z: {sec2b_df['null_sd_Z'].min():.3f} to "
      f"{sec2b_df['null_sd_Z'].max():.3f}   (nominal 1)")
n_miscentred = int((sec2b_df["null_mean_Z"] > 0).sum())
print(f"  cells with null mean Z > 0: {n_miscentred}/{len(sec2b_df)}")
print("  -> the DH pipeline over-rejects on this panel with NO lead-lag "
      "relation present. This is the load-bearing size result; it needs no "
      "bootstrap and no synthetic outcome, and it covers every lag-1 cell.")
print(f"\n  permutation rejection rate at p<{ALPHA} per cell "
      f"(nominal {ALPHA}): {perm_size_rates.min():.3f} to "
      f"{perm_size_rates.max():.3f}, median {perm_size_rates.median():.3f}")

# `size_ok` now means "DH is correctly sized on this panel", judged on the
# permutation nulls rather than on the δ=0 rejection rate. Section 5 branches
# on this. Both criteria must hold: nulls centred at 0 and rejection at nominal.
size_ok = bool(n_miscentred == 0 and perm_size_rates.max() <= 0.12)
print(f"\nsize_ok = {size_ok}  "
      f"(mis-centred cells {n_miscentred}/{len(sec2b_df)}, "
      f"max permutation rejection rate {perm_size_rates.max():.3f} vs "
      f"0.12 tolerance) — gated on permutation calibration, "
      f"not on the δ=0 rejection rate")
print("  [Historical shuffle-null headline, superseded]: per-cell "
      "shuffle-null rejection rate 0.235 to 0.405 (median 0.289) — "
      "see Section 2g for the circular-shift-primary figures and the "
      "regated size_ok.")

# --- z_pos vs p_cell disagreement -----------------------------------------
_max_zpos = sec2b_df.loc[sec2b_df["z_pos"].idxmax()]
_min_pcell = sec2b_df.iloc[0]
print(f"\nMost extreme by z_pos: {_max_zpos.weapon}x{_max_zpos.outcome} "
      f"(z_pos={_max_zpos.z_pos:.3f})")
print(f"Most extreme by p_cell: {_min_pcell.weapon}x{_min_pcell.outcome} "
      f"(p_cell={_min_pcell.p_cell:.4f}, z_pos={_min_pcell.z_pos:.3f})")
if _max_zpos.weapon != _min_pcell.weapon or _max_zpos.outcome != _min_pcell.outcome:
    print("  -> the two standardizations DISAGREE. The p_cell leader wins on "
          "tail shape, not on distance from its own null mean.")
print(f"Max z_pos across all cells: {sec2b_df['z_pos'].max():.3f} "
      f"(2-sd reference: 1.96) -> "
      f"{'no cell reaches 2 sd above its own null mean' if sec2b_df['z_pos'].max() < 1.96 else 'at least one cell exceeds 2 sd'}")

print(f"\nNominally significant cells (raw one-sided DH p<{ALPHA}): "
      f"{n_nominal_sig}")
print(f"Of those, surviving their OWN one-sided permutation null at "
      f"p<{ALPHA}: {n_percell_survivors}")
print(f"Surviving BH across all {len(sec2b_df)} lag-1 cells: {n_bh_survivors}")
print(f"Surviving Bonferroni: {n_bonf_survivors}")
print(f"Expected survivors under a global null at alpha={ALPHA} across "
      f"{len(sec2b_df)} tests: {ALPHA * len(sec2b_df):.1f}")

# --- Verdict: lead with multiplicity, not the raw count --------------------
_surv = sec2b_df[sec2b_df["nominal_sig"] & sec2b_df["survives_percell"]]
_surv_lst = ", ".join(f"{r.weapon}x{r.outcome}" for r in _surv.itertuples()) or "none"
_exp = ALPHA * len(sec2b_df)
_min_q = float(sec2b_df["q_bh"].min())

if n_bh_survivors == 0:
    sec2b_verdict = (
        f"RECONCILED — {n_percell_survivors} of {n_nominal_sig} nominally "
        f"significant cells exceed their own one-sided permutation null "
        f"({_surv_lst}), against {_exp:.1f} expected by chance across "
        f"{len(sec2b_df)} pre-specified lag-1 tests. No cell survives BH "
        f"correction over that family (smallest q={_min_q:.3f}) and none "
        f"reaches 2 sd above its own null mean (max z_pos="
        f"{sec2b_df['z_pos'].max():.2f}). "
        f"The count-level excess (lag-1 count p={emp_p_lag1:.4f} at "
        f"{N_PERM_2B:,} perms) is characterised in Sections 2e (omnibus) and "
        f"2f (circular-shift null), not attributed to cross-cell dependence; "
        f"it is not a nameable per-cell effect. Per-cell falsification, "
        f"which is correctly calibrated, finds nothing.")
elif n_bh_survivors < n_nominal_sig:
    _bhl = ", ".join(f"{r.weapon}x{r.outcome} (q={r.q_bh:.3f})"
                     for r in sec2b_df[sec2b_df["survives_bh"]].itertuples())
    sec2b_verdict = (
        f"PARTIAL — {n_bh_survivors} of {n_nominal_sig} nominally significant "
        f"cells survive both their own one-sided permutation null and BH "
        f"correction across the {len(sec2b_df)}-cell lag-1 family: {_bhl}. "
        f"Report these individually alongside their NB06 placebo rates. The "
        f"remaining nominal hits sit inside their own permutation "
        f"distributions.")
else:
    sec2b_verdict = (
        f"NOT reconciled — every nominally significant cell survives both its "
        f"own permutation null and BH correction. The RQ2 null cannot rest on "
        f"per-cell falsification and must be re-argued.")

print(f"\nVerdict: {sec2b_verdict}")

sec2b_df.to_csv(TBL_DIR / "section2b_percell_permutation.csv", index=False)
_size = sec2b_df[["weapon", "outcome", "null_mean_Z", "null_sd_Z",
                  "n_finite_perm", "n_nonfinite_perm",
                  "n_units_dh", "n_units_loose"]].copy()
_size["perm_reject_rate"] = [
    round(float(perm_size_rates.loc[(r.weapon, r.outcome)]), 4)
    for r in _size.itertuples()]
_size["n_perm"] = N_PERM_2B
_size.to_csv(TBL_DIR / "section2b_permutation_size.csv", index=False)

print(f"\n[Section 2b] Per-cell reconciliation done — saved → "
      f"section2b_percell_permutation.csv + section2b_permutation_size.csv")

=== Section 2b — per-cell permutation test (lag 1, 12 cells, 2,000 permutations, ONE-SIDED upper tail) ===

  weapon                 outcome   obs_Z  obs_p  null_mean_Z  null_sd_Z   z_pos  p_cell   q_bh  p_bonf  n_units_dh  n_finite_perm  nominal_sig  survives_percell  survives_bh
  GROUND              part_n_war  6.4233 0.0000       2.2280     7.3997  0.5670 0.04198 0.2024  0.5038         123           2000         True              True        False
MISSILES            part_n_minor  3.6372 0.0001       0.8858     1.5480  1.7774 0.05547 0.2024  0.6656         115           2000         True             False        False
   NAVAL            part_n_minor  3.2888 0.0005       0.7655     1.5333  1.6457 0.06147 0.2024  0.7376         104           2000         True             False        False
  GROUND part_n_extraterritorial  4.9274 0.0000       2.3930     7.2740  0.3484 0.06747 0.2024  0.8096         132           2000         True             False        False
   NAVAL part_n_extrat

D:\post graduate\assignments and projects\sem 3\BDA\PeaceMakers' Arms Race\version 2\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
D:\post graduate\assignments and projects\sem 3\BDA\PeaceMakers' Arms Race\version 2\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]



  Methods note: DH's internal min_obs filter counts observations, not variation, so a country contributing twelve zeros clears the filter and then produces an exploding Wald statistic. This is a general caveat for applying DH to sparse count outcomes, independent of this paper's findings.

=== Outcome degeneracy vs null dispersion ===
  weapon                 outcome  n_eligible  n_zero_var_outcome  frac_zero_var  wald_p99   wald_max  null_sd_Z  n_units_dh
  GROUND              part_n_war         192                  63       0.328125  0.619651 266.840753     7.3997         123
  GROUND part_n_extraterritorial         192                  55       0.286458  0.491881   0.560567     7.2740         132
MISSILES part_n_extraterritorial         192                  55       0.286458  0.308694   0.902951     2.5477         110
MISSILES              part_n_war         192                  63       0.328125  0.332299   1.063258     2.4158         106
   NAVAL part_n_extraterritorial         1


Evidence 2 — treatment-treatment correlation, observed vs permuted:
  observed mean|r| across 4 treatment columns: 0.073 (range +0.036 to +0.110)
  after within-country independent shuffling (50 draws): mean|r| = 0.012 (sd 0.0047)

Conclusion: BOTH legs of the cross-cell-dependence explanation FAIL.
  Leg 1 — the 12 cells are near-INDEPENDENT: mean|r|=0.068, median r=+0.000, M_eff=11.69 of 12. The same-weapon subset (mean|r|=0.287, n=24) is real but confined to 3-cell blocks and cannot inflate a 12-cell count null twofold.
  Leg 2 — there was no cross-treatment correlation to destroy: observed mean|r|=0.073 is negligible in ABSOLUTE terms, whatever its ratio to the permuted 0.012. A ratio test against a near-zero baseline is uninformative and the earlier guard on this comparison was invalid.

NEGATIVE RESULT — the count excess (p=0.0265) is NOT explained by cross-cell dependence. The 12 cells are near-independent (M_eff=11.69/12) and observed cross-treatment correlation is negligible 

In [8]:
# Figure: per-cell permutation null band + observed Z, 3-state colour + q_bh
fig, ax = plt.subplots(figsize=(9, 7))
labels = [f"{r.weapon}×{r.outcome}" for r in sec2b_df.itertuples()]
seen = set()
for i, r in enumerate(sec2b_df.itertuples()):
    m_, s_ = r.null_mean_Z, r.null_sd_Z
    ax.fill_betweenx([i - 0.35, i + 0.35], m_ - 2 * s_, m_ + 2 * s_,
                     color="steelblue", alpha=0.18,
                     label="permutation null mean ± 2sd" if i == 0 else None)
    ax.hlines(m_, i - 0.35, i + 0.35, color="steelblue", lw=1.0)
    state = "bh" if r.survives_bh else ("nom" if r.nominal_sig else "ns")
    lbl = None
    if state not in seen:
        lbl = {"bh": "survives BH", "nom": "nominal only (BH-rejected)",
               "ns": "not nominal"}[state]
        seen.add(state)
    if state == "bh":
        ax.scatter([r.obs_Z], [i], s=55, zorder=3, color="indianred", label=lbl)
    elif state == "nom":
        ax.scatter([r.obs_Z], [i], s=55, zorder=3, facecolors="none",
                   edgecolors="indianred", linewidths=1.5, label=lbl)
    else:
        ax.scatter([r.obs_Z], [i], s=45, zorder=3, color="steelblue", label=lbl)
    ax.annotate(f"q={r.q_bh:.2f}", xy=(0.995, i),
                xycoords=("axes fraction", "data"),
                ha="right", va="center", fontsize=7, color="dimgray")
ax.axvline(0, color="gray", ls="--", lw=0.8)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("lag-1 Z statistic", fontsize=9)
ax.set_title(f"Per-cell permutation, lag 1 ({N_PERM_2B:,} perms, one-sided): "
             f"observed Z vs each cell's own null", fontsize=10)
ax.legend(fontsize=8, loc="lower left")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig4_percell_permutation.png", dpi=300, bbox_inches="tight")
plt.close(fig)

print(f"\n[Section 2b] Figure saved → fig4_percell_permutation.png")


[Section 2b] Figure saved → fig4_percell_permutation.png


## Section 2e — Omnibus Tests on the 12 p_cell Values

BH answers "can any cell be named?" (no). It does **not** answer "is the family as
a whole anomalous?" Simes (sensitive to one strong cell) and Fisher (sensitive to
many mildly small p-values) do. Each is reported asymptotically and
**permutation-calibrated** against leave-one-out empirical p-values built from the
2000 draws, so the calibrated version inherits the cells' real dependence and the
discreteness of an empirical p.

In [9]:
# ===================== Section 2e — omnibus tests ===========================
# BH answers "can any cell be named?" (no). It does NOT answer "is the family
# as a whole anomalous?" These do. Simes is sensitive to one strong cell;
# Fisher to many mildly small p-values. Which one fires is diagnostic.
print("\n" + "=" * 74)
print("Section 2e — omnibus tests on the 12 lag-1 p_cell values")
print("=" * 74)

_pv = sec2b_df["p_cell"].to_numpy(dtype=float)
_M = len(_pv)

def fisher_stat(p):
    return float(-2.0 * np.sum(np.log(np.clip(p, 1e-300, 1.0))))

def simes_p(p):
    ps = np.sort(p)
    return float(np.min(ps * len(ps) / np.arange(1, len(ps) + 1)))

_fisher_obs = fisher_stat(_pv)
_fisher_asym = float(stats.chi2.sf(_fisher_obs, df=2 * _M))
_simes_obs = simes_p(_pv)
_n_below = int((_pv < ALPHA).sum())
_binom_p = float(stats.binomtest(_n_below, _M, ALPHA,
                                 alternative="greater").pvalue)
_ks = stats.kstest(_pv, "uniform")

print(f"\n  p_cell values (sorted): "
      + ", ".join(f"{v:.4f}" for v in np.sort(_pv)))
print(f"\n  Fisher  -2*sum(ln p) = {_fisher_obs:.3f} on {2*_M} df "
      f"-> asymptotic p = {_fisher_asym:.4f}")
print(f"  Simes global p       = {_simes_obs:.4f} "
      f"(equals smallest q_bh = {sec2b_df['q_bh'].min():.4f} by construction)")
print(f"  Binomial: {_n_below}/{_M} cells with p_cell<{ALPHA} "
      f"-> p = {_binom_p:.4f}")
print(f"  KS vs Uniform(0,1): D={_ks.statistic:.4f}, p={_ks.pvalue:.4f}")

# --- Permutation-calibrated omnibus -----------------------------------------
# For each permutation b, form leave-one-out p-values for its 12 cells against
# the OTHER 1999 draws, then combine. This calibrates Fisher and Simes against
# the actual joint null, inheriting the cells' real dependence and the
# discreteness of a 2000-draw empirical p.
_cells_ord = list(zip(sec2b_df["weapon"], sec2b_df["outcome"]))
_Zmat = np.column_stack([
    _p1_2b[(_p1_2b["weapon"] == w) & (_p1_2b["outcome"] == o)]
        .sort_values("perm")["Z"].to_numpy(dtype=float)
    for w, o in _cells_ord])
assert _Zmat.shape == (N_PERM_2B, _M), f"Z matrix shape {_Zmat.shape}"
assert np.isfinite(_Zmat).all(), "non-finite Z in the permutation matrix"

_sorted_cols = [np.sort(_Zmat[:, j]) for j in range(_M)]
_p_loo = np.empty_like(_Zmat)
for j in range(_M):
    sc = _sorted_cols[j]
    # count of OTHER draws >= this draw (self excluded)
    ge = (N_PERM_2B - np.searchsorted(sc, _Zmat[:, j], side="left")) - 1
    _p_loo[:, j] = (1 + ge) / (1 + (N_PERM_2B - 1))

_fisher_null = np.array([fisher_stat(_p_loo[b]) for b in range(N_PERM_2B)])
_simes_null = np.array([simes_p(_p_loo[b]) for b in range(N_PERM_2B)])
_fisher_perm_p = (1 + int((_fisher_null >= _fisher_obs).sum())) / (1 + N_PERM_2B)
_simes_perm_p = (1 + int((_simes_null <= _simes_obs).sum())) / (1 + N_PERM_2B)

print(f"\n  permutation-CALIBRATED (leave-one-out, {N_PERM_2B:,} draws):")
print(f"    Fisher: observed {_fisher_obs:.3f} vs null mean "
      f"{_fisher_null.mean():.3f} (sd {_fisher_null.std():.3f}) "
      f"-> p = {_fisher_perm_p:.4f}")
print(f"    Simes:  observed {_simes_obs:.4f} vs null median "
      f"{np.median(_simes_null):.4f} -> p = {_simes_perm_p:.4f}")

_fisher_sig = _fisher_perm_p < ALPHA
_simes_sig = _simes_perm_p < ALPHA
if _fisher_sig and not _simes_sig:
    omnibus_reading = (
        f"DIFFUSE aggregate excess. Fisher rejects (p={_fisher_perm_p:.4f}) "
        f"while Simes does not (p={_simes_perm_p:.4f}) — the signature of "
        f"several mildly small p-values rather than one strong cell, which is "
        f"exactly the observed pattern (four cells in 0.042-0.068, then a gap "
        f"to 0.242). Consistent with the count test (p={emp_p_lag1:.4f}). "
        f"'0/12 survive BH' and 'the family is mildly anomalous' are both "
        f"true and not in conflict: BH controls FDR among rejections, it is "
        f"not a global test.")
elif _simes_sig:
    omnibus_reading = (
        f"CONCENTRATED excess — Simes rejects (p={_simes_perm_p:.4f}), so the "
        f"anomaly is carried by the strongest cell. Reconcile against BH "
        f"explicitly before writing this up; the two should not normally "
        f"disagree in this direction.")
else:
    omnibus_reading = (
        f"NO aggregate excess under either omnibus test "
        f"(Fisher p={_fisher_perm_p:.4f}, Simes p={_simes_perm_p:.4f}). The "
        f"count test's p={emp_p_lag1:.4f} is then the outlier among the "
        f"aggregate diagnostics and should be reported as such rather than as "
        f"the headline aggregate result.")
print(f"\n  Reading: {omnibus_reading}")

pd.DataFrame([
    {"test": "fisher_stat", "value": _fisher_obs},
    {"test": "fisher_p_asymptotic", "value": _fisher_asym},
    {"test": "fisher_p_permutation", "value": _fisher_perm_p},
    {"test": "simes_p", "value": _simes_obs},
    {"test": "simes_p_permutation", "value": _simes_perm_p},
    {"test": "binomial_p", "value": _binom_p},
    {"test": "ks_D", "value": float(_ks.statistic)},
    {"test": "ks_p", "value": float(_ks.pvalue)},
    {"test": "n_below_alpha", "value": _n_below},
    {"test": "count_test_p", "value": emp_p_lag1},
    {"test": "n_perm", "value": N_PERM_2B},
]).to_csv(TBL_DIR / "section2e_omnibus.csv", index=False)
print("[Section 2e] omnibus tests saved -> section2e_omnibus.csv")


Section 2e — omnibus tests on the 12 lag-1 p_cell values

  p_cell values (sorted): 0.0420, 0.0555, 0.0615, 0.0675, 0.2419, 0.2504, 0.2994, 0.3188, 0.4478, 0.5387, 0.6812, 0.7221

  Fisher  -2*sum(ln p) = 37.665 on 24 df -> asymptotic p = 0.0375
  Simes global p       = 0.2024 (equals smallest q_bh = 0.2024 by construction)
  Binomial: 1/12 cells with p_cell<0.05 -> p = 0.4596
  KS vs Uniform(0,1): D=0.3478, p=0.0842

  permutation-CALIBRATED (leave-one-out, 2,000 draws):
    Fisher: observed 37.665 vs null mean 23.943 (sd 8.607) -> p = 0.0705
    Simes:  observed 0.2024 vs null median 0.5580 -> p = 0.1869

  Reading: NO aggregate excess under either omnibus test (Fisher p=0.0705, Simes p=0.1869). The count test's p=0.0265 is then the outlier among the aggregate diagnostics and should be reported as such rather than as the headline aggregate result.
[Section 2e] omnibus tests saved -> section2e_omnibus.csv


## Section 2f — Circular-Shift Permutation Null

The substantive robustness test. A circular rotation preserves each treatment
series' autocorrelation function exactly while destroying its alignment with the
outcome. Only the finite values are rotated in place, keeping the NaN mask fixed
so the comparison isolates serial structure. If the count excess vanishes here, the
aggregate anomaly is a persistence artifact of the shuffle null — the lag-1
counterpart of the Section 3 differencing artifact.

In [10]:
# =============== Section 2f — circular-shift permutation null ================
N_PERM_CS = 2000

def run_perm_task_circular(perm_idx, cdata, treat_cols, grid_cells, seed):
    """One circular-shift permutation, lag-1 cells only.

    Each country's treatment series is ROTATED by a random non-zero offset
    rather than shuffled. This preserves the series' autocorrelation function
    exactly while destroying alignment with the outcome. Only the finite
    values are rotated, in place, so the missingness mask stays fixed and
    aligned with the outcome (mirroring run_perm_task_lag1).
    """
    rng = np.random.default_rng([seed, 700_000 + perm_idx])
    shifted = {}
    for iso3, arrs in cdata.items():
        new = dict(arrs)
        for tc in treat_cols:
            x = arrs[tc].copy()
            valid = np.isfinite(x)
            nv = int(valid.sum())
            if nv > 2:
                k = int(rng.integers(1, nv))       # non-zero offset
                x[valid] = np.roll(x[valid], k)
            new[tc] = x
        shifted[iso3] = new
    out = []
    for (w, o, lag) in grid_cells:
        Z, p, _ = dumitrescu_hurlin_fast(shifted, f"d_log_tiv_{w}",
                                         f"d_log_{o}", lag)
        out.append({"perm": perm_idx, "weapon": w, "outcome": o, "lag": lag,
                    "Z": Z, "p": p})
    return out

PERMCS_CACHE = CACHE_DIR / f"nb12_perm_circshift_{N_PERM_CS}.parquet"
_exp_cs = N_PERM_CS * len(LAG1_GRID)
cache_ok_cs = False
if PERMCS_CACHE.exists():
    permcs_df = pd.read_parquet(PERMCS_CACHE)
    cache_ok_cs = (len(permcs_df) == _exp_cs
                   and permcs_df["perm"].nunique() == N_PERM_CS
                   and set(zip(permcs_df["weapon"], permcs_df["outcome"]))
                       == set(LAG1_CELLS))
    if cache_ok_cs and "panel_fp" in permcs_df.columns:
        if permcs_df["panel_fp"].iloc[0] != PANEL_FP:
            print("  circular-shift cache fingerprint MISMATCH — MISS")
            cache_ok_cs = False
if cache_ok_cs:
    print(f"Cache HIT — loaded {len(permcs_df):,} circular-shift results "
          f"from {PERMCS_CACHE.name}")
else:
    print(f"Cache MISS — running {N_PERM_CS:,} circular shifts x "
          f"{len(LAG1_GRID)} cells = {_exp_cs:,} DH calls")
    t0 = time.perf_counter()
    _res_cs = Parallel(n_jobs=-2)(
        delayed(run_perm_task_circular)(i, country_data, TREAT_COLS,
                                        LAG1_GRID, SEED)
        for i in range(N_PERM_CS))
    permcs_df = pd.DataFrame([r for rows in _res_cs for r in rows])
    permcs_df["panel_fp"] = PANEL_FP
    permcs_df.to_parquet(PERMCS_CACHE, index=False)
    print(f"Computed fresh in {time.perf_counter() - t0:.0f}s -> "
          f"{PERMCS_CACHE.name}")

# --- ACF preservation check: the whole point of the scheme -------------------
def _lag1_acf(v):
    v = v[np.isfinite(v)]
    if v.size < 4 or np.std(v) == 0:
        return np.nan
    return float(np.corrcoef(v[:-1], v[1:])[0, 1])

_acf_obs, _acf_shuf, _acf_circ = [], [], []
_rng_acf = np.random.default_rng([SEED, 800_000])
for arrs in country_data.values():
    for tc in TREAT_COLS:
        x = arrs[tc]
        v = x[np.isfinite(x)]
        if v.size < 4:
            continue
        _acf_obs.append(_lag1_acf(v))
        _acf_shuf.append(_lag1_acf(_rng_acf.permutation(v)))
        _acf_circ.append(_lag1_acf(np.roll(v, int(_rng_acf.integers(1, v.size)))))
_acf = {k: float(np.nanmean(a)) for k, a in
        [("observed", _acf_obs), ("shuffled", _acf_shuf),
         ("circular", _acf_circ)]}
print(f"\n  ACF check — mean lag-1 autocorrelation of the treatment series:")
print(f"    observed {_acf['observed']:+.4f} | within-country shuffle "
      f"{_acf['shuffled']:+.4f} | circular shift {_acf['circular']:+.4f}")
_acf_preserved = (abs(_acf["circular"] - _acf["observed"])
                  < 0.5 * abs(_acf["observed"] - _acf["shuffled"]))
print(f"    -> circular shift {'PRESERVES' if _acf_preserved else 'does NOT preserve'} "
      f"serial structure that the shuffle destroys")

# --- Per-cell and count results under the circular-shift null ---------------
_p1_cs = permcs_df[permcs_df["lag"] == 1]
cs_rows = []
for w, o in LAG1_CELLS:
    _obs_Z = float(sec2b_df.loc[(sec2b_df["weapon"] == w)
                                & (sec2b_df["outcome"] == o), "obs_Z"].iloc[0])
    nz = _p1_cs[(_p1_cs["weapon"] == w) & (_p1_cs["outcome"] == o)]["Z"].to_numpy()
    nzf = nz[np.isfinite(nz)]
    cs_rows.append({
        "weapon": w, "outcome": o, "obs_Z": round(_obs_Z, 4),
        "cs_null_mean_Z": round(float(np.mean(nzf)), 4),
        "cs_null_sd_Z": round(float(np.std(nzf)), 4),
        "cs_n_finite": int(nzf.size),
        "p_cell_cs": round((1 + int((nzf >= _obs_Z).sum())) / (1 + nzf.size), 5),
    })
cs_df = pd.DataFrame(cs_rows)
cs_df["q_bh_cs"] = np.round(
    multipletests(cs_df["p_cell_cs"].values, alpha=ALPHA,
                  method="fdr_bh")[1], 4)
cs_df = cs_df.merge(sec2b_df[["weapon", "outcome", "p_cell", "q_bh",
                              "null_mean_Z", "null_sd_Z", "nominal_sig"]],
                    on=["weapon", "outcome"]).sort_values("p_cell_cs")

print(f"\n=== Circular-shift vs shuffle, per cell ===")
print(cs_df[["weapon", "outcome", "obs_Z", "null_mean_Z", "cs_null_mean_Z",
             "null_sd_Z", "cs_null_sd_Z", "p_cell", "p_cell_cs",
             "q_bh", "q_bh_cs", "nominal_sig"]].to_string(index=False))

_cs_counts = (_p1_cs.assign(sig=_p1_cs["p"] < ALPHA)
                    .groupby("perm")["sig"].sum())
_cs_exceed = int((_cs_counts >= obs_count_lag1).sum())
emp_p_lag1_cs = (1 + _cs_exceed) / (1 + len(_cs_counts))
_cs_rates = (_p1_cs.groupby(["weapon", "outcome"])["p"]
                   .apply(lambda s: float(np.mean(s.dropna() < ALPHA))))

print(f"\n=== Count test under both nulls (observed = {obs_count_lag1}/12) ===")
print(f"  shuffle null:         mean={null_counts_lag1.mean():.3f} "
      f"(sd={null_counts_lag1.std():.3f}), p={emp_p_lag1:.4f}")
print(f"  circular-shift null:  mean={_cs_counts.mean():.3f} "
      f"(sd={_cs_counts.std():.3f}), p={emp_p_lag1_cs:.4f}")
print(f"  per-cell rejection rate at p<{ALPHA}: shuffle "
      f"{perm_size_rates.min():.3f}-{perm_size_rates.max():.3f} "
      f"(median {perm_size_rates.median():.3f}) vs circular "
      f"{_cs_rates.min():.3f}-{_cs_rates.max():.3f} "
      f"(median {_cs_rates.median():.3f})")
print(f"  null mean Z: shuffle {sec2b_df['null_mean_Z'].min():+.3f} to "
      f"{sec2b_df['null_mean_Z'].max():+.3f} vs circular "
      f"{cs_df['cs_null_mean_Z'].min():+.3f} to "
      f"{cs_df['cs_null_mean_Z'].max():+.3f}")
_cs_bh = int((cs_df["q_bh_cs"] < ALPHA).sum())
_cs_percell = int((cs_df["nominal_sig"] & (cs_df["p_cell_cs"] < ALPHA)).sum())
print(f"  survivors: per-cell {_cs_percell}/{n_nominal_sig} "
      f"(shuffle: {n_percell_survivors}), BH {_cs_bh}/12 "
      f"(shuffle: {n_bh_survivors})")

_looser = _cs_counts.mean() > null_counts_lag1.mean()
if emp_p_lag1_cs >= 0.10 and _looser:
    cs_reading = (
        f"CONFIRMED persistence artifact. Preserving the treatments' serial "
        f"structure raises the null count from "
        f"{null_counts_lag1.mean():.2f} to {_cs_counts.mean():.2f} and the "
        f"count excess disappears (p={emp_p_lag1:.4f} -> "
        f"{emp_p_lag1_cs:.4f}). The shuffle null was too lenient because it "
        f"generated white-in-time regressors while the observed data are "
        f"serially structured, and DH's over-rejection is worse with "
        f"persistent regressors — the lag-1 counterpart of the Section 3 "
        f"differencing artifact. The RQ2 null is clean at aggregate AND "
        f"per-cell level; report the circular-shift null as primary.")
elif emp_p_lag1_cs < ALPHA:
    cs_reading = (
        f"NOT an artifact. The excess survives a serial-structure-preserving "
        f"null (p={emp_p_lag1_cs:.4f} vs {emp_p_lag1:.4f}). Report a genuine "
        f"diffuse aggregate excess that no individual cell accounts for "
        f"({_cs_bh}/12 survive BH under this null), and let the multiplicity "
        f"result carry the no-nameable-effect conclusion. Do NOT claim the "
        f"aggregate is clean.")
else:
    cs_reading = (
        f"PARTIAL — the excess weakens but does not clear "
        f"(p={emp_p_lag1:.4f} -> {emp_p_lag1_cs:.4f}, null count "
        f"{null_counts_lag1.mean():.2f} -> {_cs_counts.mean():.2f}). Serial "
        f"structure accounts for part of it. Report both nulls side by side "
        f"and rest the conclusion on the per-cell and multiplicity results.")
print(f"\n  Reading: {cs_reading}")

cs_df.to_csv(TBL_DIR / "section2f_circular_shift.csv", index=False)
pd.DataFrame([
    {"metric": "obs_count_lag1", "value": obs_count_lag1},
    {"metric": "shuffle_null_mean", "value": null_counts_lag1.mean()},
    {"metric": "shuffle_null_sd", "value": null_counts_lag1.std()},
    {"metric": "shuffle_p", "value": emp_p_lag1},
    {"metric": "circshift_null_mean", "value": _cs_counts.mean()},
    {"metric": "circshift_null_sd", "value": _cs_counts.std()},
    {"metric": "circshift_p", "value": emp_p_lag1_cs},
    {"metric": "acf_observed", "value": _acf["observed"]},
    {"metric": "acf_shuffled", "value": _acf["shuffled"]},
    {"metric": "acf_circular", "value": _acf["circular"]},
    {"metric": "n_perm", "value": N_PERM_CS},
]).to_csv(TBL_DIR / "section2f_null_comparison.csv", index=False)
print("[Section 2f] circular-shift results saved -> "
      "section2f_circular_shift.csv + section2f_null_comparison.csv")

Cache HIT — loaded 24,000 circular-shift results from nb12_perm_circshift_2000.parquet


D:\post graduate\assignments and projects\sem 3\BDA\PeaceMakers' Arms Race\version 2\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
D:\post graduate\assignments and projects\sem 3\BDA\PeaceMakers' Arms Race\version 2\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]



  ACF check — mean lag-1 autocorrelation of the treatment series:
    observed -0.3286 | within-country shuffle -0.0232 | circular shift -0.3113
    -> circular shift PRESERVES serial structure that the shuffle destroys

=== Circular-shift vs shuffle, per cell ===
  weapon                 outcome   obs_Z  null_mean_Z  cs_null_mean_Z  null_sd_Z  cs_null_sd_Z  p_cell  p_cell_cs   q_bh  q_bh_cs  nominal_sig
  GROUND              part_n_war  6.4233       2.2280          2.3790     7.3997        6.7108 0.04198    0.03248 0.2024   0.3898         True
  GROUND part_n_extraterritorial  4.9274       2.3930          2.2387     7.2740        6.5663 0.06747    0.06997 0.2024   0.4198         True
MISSILES            part_n_minor  3.6372       0.8858          1.3982     1.5480        1.8326 0.05547    0.11994 0.2024   0.4318         True
   NAVAL            part_n_minor  3.2888       0.7655          1.3271     1.5333        1.7895 0.06147    0.14393 0.2024   0.4318         True
  GROUND           

In [11]:
# Figure: null-scheme comparison (count histograms + per-cell centring)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
_bmax = max(null_counts_lag1.max(), _cs_counts.max(), obs_count_lag1)
_bins = np.arange(-0.5, _bmax + 1.5, 1)
ax.hist(null_counts_lag1, bins=_bins, alpha=0.55, color="steelblue",
        edgecolor="white", label=f"shuffle null (p={emp_p_lag1:.4f})")
ax.hist(_cs_counts, bins=_bins, alpha=0.55, color="seagreen",
        edgecolor="white", label=f"circular-shift null (p={emp_p_lag1_cs:.4f})")
ax.axvline(obs_count_lag1, color="indianred", lw=1.8,
           label=f"observed = {obs_count_lag1}")
ax.set_xlabel("significant lag-1 cells (p<0.05) out of 12", fontsize=9)
ax.set_ylabel("permutations", fontsize=9)
ax.set_title("Count-test null: shuffle vs circular-shift", fontsize=10)
ax.legend(fontsize=8)

ax = axes[1]
_ord = cs_df.sort_values("null_mean_Z", ascending=False).reset_index(drop=True)
_yy = np.arange(len(_ord))
ax.scatter(_ord["null_mean_Z"], _yy, color="steelblue", zorder=3,
           label="shuffle null mean Z")
ax.scatter(_ord["cs_null_mean_Z"], _yy, color="seagreen", marker="s", zorder=3,
           label="circular-shift null mean Z")
ax.axvline(0, color="gray", ls="--", lw=0.8)
ax.set_yticks(_yy)
ax.set_yticklabels([f"{r.weapon}×{r.outcome}" for r in _ord.itertuples()],
                   fontsize=7)
ax.invert_yaxis()
ax.set_xlabel("per-cell null mean Z", fontsize=9)
ax.set_title("Per-cell null centring by scheme", fontsize=10)
ax.legend(fontsize=8, loc="lower right")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig5_null_scheme_comparison.png", dpi=300,
            bbox_inches="tight")
plt.close(fig)
print("[Section 2f] Figure saved -> fig5_null_scheme_comparison.png")

[Section 2f] Figure saved -> fig5_null_scheme_comparison.png


## Section 2g — Recalibrated Omnibus on the Circular-Shift Null; Near-Duplicate Pair

Three loose ends from Section 2f. (1) The Section 2e omnibus tests were computed on the shuffle-based `p_cell`, but Section 2f showed the circular-shift null is the better-specified reference — recompute Fisher/Simes on `p_cell_cs`. (2) The `max|r|=0.951` pair from the cross-cell dependence measurement (Section 2b) was never identified; find it and test whether collapsing it to one test moves the aggregate picture. (3) Explain the dispersion asymmetry noted in Section 2f: the circular shift *tightened* both GROUND cells' nulls while widening nearly every other cell's.

In [12]:
# =============== Section 2g — recalibrated omnibus (circular-shift null) ====
print("\n" + "=" * 74)
print("Section 2g — Task A: omnibus tests recalibrated on the circular-shift null")
print("=" * 74)

_cells_ord_cs = list(zip(cs_df["weapon"], cs_df["outcome"]))
_pv_cs = cs_df["p_cell_cs"].to_numpy(dtype=float)
_M_cs = len(_pv_cs)
assert _M_cs == 12, f"expected 12 circular-shift cells, got {_M_cs}"

_fisher_obs_cs = fisher_stat(_pv_cs)
_fisher_asym_cs = float(stats.chi2.sf(_fisher_obs_cs, df=2 * _M_cs))
_simes_obs_cs = simes_p(_pv_cs)

_Zmat_cs = np.column_stack([
    _p1_cs[(_p1_cs["weapon"] == w) & (_p1_cs["outcome"] == o)]
        .sort_values("perm")["Z"].to_numpy(dtype=float)
    for w, o in _cells_ord_cs])
assert _Zmat_cs.shape == (N_PERM_CS, _M_cs), f"Z matrix shape {_Zmat_cs.shape}"
assert np.isfinite(_Zmat_cs).all(), "non-finite Z in the circular-shift matrix"

_sorted_cols_cs = [np.sort(_Zmat_cs[:, j]) for j in range(_M_cs)]
_p_loo_cs = np.empty_like(_Zmat_cs)
for j in range(_M_cs):
    sc = _sorted_cols_cs[j]
    ge = (N_PERM_CS - np.searchsorted(sc, _Zmat_cs[:, j], side="left")) - 1
    _p_loo_cs[:, j] = (1 + ge) / (1 + (N_PERM_CS - 1))

_fisher_null_cs = np.array([fisher_stat(_p_loo_cs[b]) for b in range(N_PERM_CS)])
_simes_null_cs = np.array([simes_p(_p_loo_cs[b]) for b in range(N_PERM_CS)])
_fisher_cs_perm_p = (1 + int((_fisher_null_cs >= _fisher_obs_cs).sum())) / (1 + N_PERM_CS)
_simes_cs_perm_p = (1 + int((_simes_null_cs <= _simes_obs_cs).sum())) / (1 + N_PERM_CS)

_n_below_cs = int((_pv_cs < ALPHA).sum())
_binom_p_cs = float(stats.binomtest(_n_below_cs, _M_cs, ALPHA,
                                    alternative="greater").pvalue)
_ks_cs = stats.kstest(_pv_cs, "uniform")

_fisher_null_mean_sh = float(_fisher_null.mean())
_fisher_null_sd_sh = float(_fisher_null.std())
_fisher_null_mean_cs = float(_fisher_null_cs.mean())
_fisher_null_sd_cs = float(_fisher_null_cs.std())
_vif_sd_shuffle = _fisher_null_sd_sh / np.sqrt(48)
_vif_var_shuffle = (_fisher_null_sd_sh ** 2) / 48.0
_vif_sd_cs = _fisher_null_sd_cs / np.sqrt(48)
_vif_var_cs = (_fisher_null_sd_cs ** 2) / 48.0

print(f"\n  Fisher null mean/sd — shuffle: {_fisher_null_mean_sh:.3f}/"
      f"{_fisher_null_sd_sh:.3f} (VIF sd={_vif_sd_shuffle:.3f}, "
      f"var={_vif_var_shuffle:.3f}); circular: {_fisher_null_mean_cs:.3f}/"
      f"{_fisher_null_sd_cs:.3f} (VIF sd={_vif_sd_cs:.3f}, "
      f"var={_vif_var_cs:.3f}) [theoretical: mean=2M={2 * _M}, "
      f"sd=sqrt(48)={np.sqrt(48):.3f}]")

_omni_compare = pd.DataFrame([
    {"test": "fisher_stat_obs", "shuffle": _fisher_obs, "circular_shift": _fisher_obs_cs},
    {"test": "fisher_p_asymptotic", "shuffle": _fisher_asym, "circular_shift": _fisher_asym_cs},
    {"test": "fisher_p_permutation", "shuffle": _fisher_perm_p, "circular_shift": _fisher_cs_perm_p},
    {"test": "fisher_null_mean", "shuffle": _fisher_null_mean_sh, "circular_shift": _fisher_null_mean_cs},
    {"test": "fisher_null_sd", "shuffle": _fisher_null_sd_sh, "circular_shift": _fisher_null_sd_cs},
    {"test": "fisher_vif_sd", "shuffle": _vif_sd_shuffle, "circular_shift": _vif_sd_cs},
    {"test": "fisher_vif_var", "shuffle": _vif_var_shuffle, "circular_shift": _vif_var_cs},
    {"test": "simes_p", "shuffle": _simes_obs, "circular_shift": _simes_obs_cs},
    {"test": "simes_p_permutation", "shuffle": _simes_perm_p, "circular_shift": _simes_cs_perm_p},
    {"test": "binomial_p", "shuffle": _binom_p, "circular_shift": _binom_p_cs},
    {"test": "ks_p", "shuffle": float(_ks.pvalue), "circular_shift": float(_ks_cs.pvalue)},
    {"test": "count_test_p", "shuffle": emp_p_lag1, "circular_shift": emp_p_lag1_cs},
])
print("\n=== Omnibus comparison, both nulls ===")
print(_omni_compare.to_string(index=False))

_simes_cs_identity_ok = abs(_simes_obs_cs - float(cs_df["q_bh_cs"].min())) < 2e-4
print(f"\n  Simes identity (circular-shift): {_simes_obs_cs:.4f} == smallest "
      f"q_bh_cs {cs_df['q_bh_cs'].min():.4f} -> {_simes_cs_identity_ok}")

if _fisher_cs_perm_p >= 0.10:
    omnibus_cs_reading = (
        f"The aggregate picture is unambiguous. Under the better-specified "
        f"null no aggregate diagnostic clears 0.05 (count p={emp_p_lag1_cs:.4f} "
        f"borderline, Fisher p={_fisher_cs_perm_p:.4f} clear). The "
        f"shuffle-null count excess was an artifact of a null that failed to "
        f"preserve regressor persistence, and the RQ2 null is clean at "
        f"aggregate and per-cell level.")
elif _fisher_cs_perm_p < ALPHA:
    omnibus_cs_reading = (
        f"The aggregate excess strengthens under the better null (Fisher "
        f"p={_fisher_cs_perm_p:.4f}). Report it as a genuine diffuse anomaly "
        f"and do not claim the aggregate is clean.")
else:
    omnibus_cs_reading = (
        f"Still borderline (Fisher p={_fisher_cs_perm_p:.4f}, count "
        f"p={emp_p_lag1_cs:.4f}). Report all four aggregate diagnostics (both "
        f"count tests, both Fisher calibrations) as a block and rest the "
        f"conclusion on per-cell and multiplicity.")
print(f"\n  Reading: {omnibus_cs_reading}")

_omni_compare.to_csv(TBL_DIR / "section2g_omnibus_both_nulls.csv", index=False)
print("[Section 2g] omnibus (both nulls) saved -> section2g_omnibus_both_nulls.csv")


Section 2g — Task A: omnibus tests recalibrated on the circular-shift null

  Fisher null mean/sd — shuffle: 23.943/8.607 (VIF sd=1.242, var=1.543); circular: 23.943/8.501 (VIF sd=1.227, var=1.505) [theoretical: mean=2M=24, sd=sqrt(48)=6.928]

=== Omnibus comparison, both nulls ===
                test   shuffle  circular_shift
     fisher_stat_obs 37.665156       31.321920
 fisher_p_asymptotic  0.037508        0.144716
fisher_p_permutation  0.070465        0.182909
    fisher_null_mean 23.943367       23.943367
      fisher_null_sd  8.607173        8.500519
       fisher_vif_sd  1.242338        1.226944
      fisher_vif_var  1.543405        1.505392
             simes_p  0.202410        0.389760
 simes_p_permutation  0.186907        0.362319
          binomial_p  0.459640        0.459640
                ks_p  0.084166        0.269507
        count_test_p  0.026487        0.097451

  Simes identity (circular-shift): 0.3898 == smallest q_bh_cs 0.3898 -> True

  Reading: The aggregate p

In [13]:
# ============= Section 2g — Task B: near-duplicate pair, collapse ===========
print("\n" + "=" * 74)
print("Section 2g — Task B: near-duplicate pair identification and collapse")
print("=" * 74)

WEAPON_NAMES_FOR_PARSE = sorted(set(w for w, _ in LAG1_CELLS), key=len, reverse=True)

def _parse_cell(name):
    for w in WEAPON_NAMES_FOR_PARSE:
        if name.startswith(w):
            return w, name[len(w) + 1:]
    raise ValueError(f"cannot parse cell label {name!r}")

def _top5_pairs(zcorr):
    off = zcorr.where(~np.eye(len(zcorr), dtype=bool)).stack()
    seen, rows = set(), []
    for (a, b), r in off.items():
        key = tuple(sorted((a, b)))
        if key in seen:
            continue
        seen.add(key)
        rows.append({"pair": f"{a} & {b}", "r": float(r)})
    out = pd.DataFrame(rows)
    return out.reindex(out["r"].abs().sort_values(ascending=False).index).head(5)

_zcorr_shuffle = _zcorr  # computed earlier in Section 2b's dependence block
_zmat_cs_pivot = (_p1_cs.assign(cell=_p1_cs["weapon"] + "x" + _p1_cs["outcome"])
                        .pivot(index="perm", columns="cell", values="Z"))
_zcorr_cs = _zmat_cs_pivot.corr()

_top5_shuffle = _top5_pairs(_zcorr_shuffle)
_top5_cs = _top5_pairs(_zcorr_cs)
print("\n  Top 5 correlated pairs — shuffle scheme:")
print(_top5_shuffle.to_string(index=False))
print("\n  Top 5 correlated pairs — circular-shift scheme:")
print(_top5_cs.to_string(index=False))

_ground_pair_expected = {"GROUNDxpart_n_war", "GROUNDxpart_n_extraterritorial"}
_max_pair_shuffle_set = set(_top5_shuffle.iloc[0]["pair"].split(" & "))
_max_pair_cs_set = set(_top5_cs.iloc[0]["pair"].split(" & "))
_is_ground_pair_shuffle = _max_pair_shuffle_set == _ground_pair_expected
_is_ground_pair_cs = _max_pair_cs_set == _ground_pair_expected
print(f"\n  Max-|r| pair is GROUND×part_n_war & "
      f"GROUND×part_n_extraterritorial? shuffle: "
      f"{_is_ground_pair_shuffle} ({_top5_shuffle.iloc[0]['pair']}, "
      f"r={_top5_shuffle.iloc[0]['r']:+.3f}); circular: {_is_ground_pair_cs} "
      f"({_top5_cs.iloc[0]['pair']}, r={_top5_cs.iloc[0]['r']:+.3f})")

_pair_names = _top5_shuffle.iloc[0]["pair"].split(" & ")
PAIR_A = _parse_cell(_pair_names[0])
PAIR_B = _parse_cell(_pair_names[1])
print(f"\n  Working pair (max-|r| under the shuffle scheme, where the "
      f"0.951 correlation was first measured): {PAIR_A[0]}x{PAIR_A[1]} & "
      f"{PAIR_B[0]}x{PAIR_B[1]}")

def _pair_row(cell_tuple):
    w, o = cell_tuple
    r2b = sec2b_df[(sec2b_df["weapon"] == w) & (sec2b_df["outcome"] == o)].iloc[0]
    rcs = cs_df[(cs_df["weapon"] == w) & (cs_df["outcome"] == o)].iloc[0]
    rdg = _dg[(_dg["weapon"] == w) & (_dg["outcome"] == o)].iloc[0]
    return {"weapon": w, "outcome": o,
            "null_mean_Z_shuffle": float(r2b["null_mean_Z"]),
            "null_sd_Z_shuffle": float(r2b["null_sd_Z"]),
            "cs_null_mean_Z": float(rcs["cs_null_mean_Z"]),
            "cs_null_sd_Z": float(rcs["cs_null_sd_Z"]),
            "p_cell": float(r2b["p_cell"]), "p_cell_cs": float(rcs["p_cell_cs"]),
            "n_units_dh": int(r2b["n_units_dh"]),
            "frac_zero_var": float(rdg["frac_zero_var"])}

_pair_diag = pd.DataFrame([_pair_row(PAIR_A), _pair_row(PAIR_B)])
_labA, _labB = f"{PAIR_A[0]}x{PAIR_A[1]}", f"{PAIR_B[0]}x{PAIR_B[1]}"
_pair_diag["r_shuffle"] = float(_zcorr_shuffle.loc[_labA, _labB])
_pair_diag["r_circular"] = float(_zcorr_cs.loc[_labA, _labB])
print("\n  Max-|r| pair diagnostic:")
print(_pair_diag.to_string(index=False))

_same_degenerate_set = bool(
    abs(_pair_diag["frac_zero_var"].iloc[0] - _pair_diag["frac_zero_var"].iloc[1]) < 0.05)
print(f"\n  Same degenerate country set (frac_zero_var within 0.05)? "
      f"{_same_degenerate_set}")

# --- Collapse: 11-cell family, drop the less-extreme of the pair ------------
_pcell_cs_A = float(_pair_diag.loc[0, "p_cell_cs"])
_pcell_cs_B = float(_pair_diag.loc[1, "p_cell_cs"])
DROP_CELL = PAIR_B if _pcell_cs_A <= _pcell_cs_B else PAIR_A
KEEP_CELL = PAIR_A if DROP_CELL == PAIR_B else PAIR_B
print(f"\n  Collapsing pair: keep {KEEP_CELL[0]}x{KEEP_CELL[1]} "
      f"(p_cell_cs={min(_pcell_cs_A, _pcell_cs_B):.5f}), drop "
      f"{DROP_CELL[0]}x{DROP_CELL[1]} "
      f"(p_cell_cs={max(_pcell_cs_A, _pcell_cs_B):.5f})")

_drop_label = f"{DROP_CELL[0]}x{DROP_CELL[1]}"
_cells_ord_11 = [c for c in _cells_ord_cs if f"{c[0]}x{c[1]}" != _drop_label]
_keep_idx_11 = [i for i, c in enumerate(_cells_ord_cs) if f"{c[0]}x{c[1]}" != _drop_label]
assert len(_cells_ord_11) == 11

_cs_df_idx = cs_df.set_index(["weapon", "outcome"])
_pv_cs_11 = _cs_df_idx.loc[_cells_ord_11, "p_cell_cs"].to_numpy(dtype=float)
_bh_11 = multipletests(_pv_cs_11, alpha=ALPHA, method="fdr_bh")
_q_bh_11_min = float(np.min(_bh_11[1]))

_fisher_obs_11 = fisher_stat(_pv_cs_11)
_p_loo_cs_11 = _p_loo_cs[:, _keep_idx_11]
_fisher_null_11 = np.array([fisher_stat(_p_loo_cs_11[b]) for b in range(N_PERM_CS)])
_fisher_p_11 = (1 + int((_fisher_null_11 >= _fisher_obs_11).sum())) / (1 + N_PERM_CS)

_p1_cs_11 = _p1_cs[~((_p1_cs["weapon"] == DROP_CELL[0])
                     & (_p1_cs["outcome"] == DROP_CELL[1]))]
_sec2b_idx = sec2b_df.set_index(["weapon", "outcome"])
_obs_count_11 = int(_sec2b_idx.loc[_cells_ord_11, "nominal_sig"].sum())
_null_counts_11 = (_p1_cs_11.assign(sig=_p1_cs_11["p"] < ALPHA)
                            .groupby("perm")["sig"].sum())
_count_p_11 = (1 + int((_null_counts_11 >= _obs_count_11).sum())) / (1 + len(_null_counts_11))

print(f"\n  Collapsed-11 (circular-shift null): BH smallest q={_q_bh_11_min:.4f}, "
      f"Fisher p={_fisher_p_11:.4f} (obs stat {_fisher_obs_11:.3f}, null mean "
      f"{_fisher_null_11.mean():.3f}), count observed={_obs_count_11}/11 "
      f"null mean={_null_counts_11.mean():.3f}, p={_count_p_11:.4f}")

# --- Bound: 10-cell family, both GROUND cells dropped ------------------------
_drop2_labels = {_labA, _labB}
_cells_ord_10 = [c for c in _cells_ord_cs if f"{c[0]}x{c[1]}" not in _drop2_labels]
_keep_idx_10 = [i for i, c in enumerate(_cells_ord_cs) if f"{c[0]}x{c[1]}" not in _drop2_labels]
assert len(_cells_ord_10) == 10

_pv_cs_10 = _cs_df_idx.loc[_cells_ord_10, "p_cell_cs"].to_numpy(dtype=float)
_bh_10 = multipletests(_pv_cs_10, alpha=ALPHA, method="fdr_bh")
_q_bh_10_min = float(np.min(_bh_10[1]))

_fisher_obs_10 = fisher_stat(_pv_cs_10)
_p_loo_cs_10 = _p_loo_cs[:, _keep_idx_10]
_fisher_null_10 = np.array([fisher_stat(_p_loo_cs_10[b]) for b in range(N_PERM_CS)])
_fisher_p_10 = (1 + int((_fisher_null_10 >= _fisher_obs_10).sum())) / (1 + N_PERM_CS)

_p1_cs_10 = _p1_cs[~(((_p1_cs["weapon"] == PAIR_A[0]) & (_p1_cs["outcome"] == PAIR_A[1]))
                     | ((_p1_cs["weapon"] == PAIR_B[0]) & (_p1_cs["outcome"] == PAIR_B[1])))]
_obs_count_10 = int(_sec2b_idx.loc[_cells_ord_10, "nominal_sig"].sum())
_null_counts_10 = (_p1_cs_10.assign(sig=_p1_cs_10["p"] < ALPHA)
                            .groupby("perm")["sig"].sum())
_count_p_10 = (1 + int((_null_counts_10 >= _obs_count_10).sum())) / (1 + len(_null_counts_10))

print(f"\n  Dropped-10 bound (both GROUND cells removed): BH smallest "
      f"q={_q_bh_10_min:.4f}, Fisher p={_fisher_p_10:.4f}, count "
      f"observed={_obs_count_10}/10 null mean={_null_counts_10.mean():.3f}, "
      f"p={_count_p_10:.4f}")

if _fisher_p_11 >= 0.10 and _count_p_11 >= 0.10:
    pair_collapse_reading = (
        f"The residual aggregate excess is attributable to a single "
        f"near-duplicate cell pair sharing a degenerate country set: "
        f"{KEEP_CELL[0]}x{KEEP_CELL[1]} & {DROP_CELL[0]}x{DROP_CELL[1]}. It "
        f"is one effective test rather than two, and its own q_bh_cs "
        f"({float(_cs_df_idx.loc[KEEP_CELL, 'q_bh_cs']):.4f}) is far from "
        f"significance.")
else:
    pair_collapse_reading = (
        f"The excess is not localised to the pair (collapsed-11 Fisher "
        f"p={_fisher_p_11:.4f}, count p={_count_p_11:.4f}). Report the "
        f"collapsed values honestly alongside the full-family ones and do "
        f"not attribute.")
print(f"\n  Reading: {pair_collapse_reading}")

with open(TBL_DIR / "section2g_pair_collapse.csv", "w", newline="") as _f:
    _f.write("# pair diagnostics\n")
    _pair_diag.to_csv(_f, index=False)
    _f.write("\n# collapsed aggregates\n")
    pd.DataFrame([
        {"family": "full_12", "n_cells": 12,
         "bh_min_q": float(cs_df["q_bh_cs"].min()),
         "fisher_p": _fisher_cs_perm_p, "count_p": emp_p_lag1_cs},
        {"family": "collapsed_11", "n_cells": 11, "bh_min_q": _q_bh_11_min,
         "fisher_p": _fisher_p_11, "count_p": _count_p_11},
        {"family": "dropped_10", "n_cells": 10, "bh_min_q": _q_bh_10_min,
         "fisher_p": _fisher_p_10, "count_p": _count_p_10},
    ]).to_csv(_f, index=False)
print("[Section 2g] pair diagnostics + collapsed aggregates saved -> "
      "section2g_pair_collapse.csv")


Section 2g — Task B: near-duplicate pair identification and collapse

  Top 5 correlated pairs — shuffle scheme:
                                                    pair        r
      GROUNDxpart_n_extraterritorial & GROUNDxpart_n_war 0.951139
      NAVALxpart_n_extraterritorial & NAVALxpart_n_minor 0.557531
          AIRxpart_n_extraterritorial & AIRxpart_n_minor 0.498853
MISSILESxpart_n_extraterritorial & MISSILESxpart_n_minor 0.427458
            AIRxpart_n_extraterritorial & AIRxpart_n_war 0.305590

  Top 5 correlated pairs — circular-shift scheme:
                                                    pair        r
      GROUNDxpart_n_extraterritorial & GROUNDxpart_n_war 0.944702
      NAVALxpart_n_extraterritorial & NAVALxpart_n_minor 0.512425
MISSILESxpart_n_extraterritorial & MISSILESxpart_n_minor 0.404154
          AIRxpart_n_extraterritorial & AIRxpart_n_minor 0.366603
            AIRxpart_n_extraterritorial & AIRxpart_n_war 0.288232

  Max-|r| pair is GROUND×part_n_war & GROU


  Collapsed-11 (circular-shift null): BH smallest q=0.3573, Fisher p=0.2784 (obs stat 26.003, null mean 21.948), count observed=7/11 null mean=4.495, p=0.1404

  Dropped-10 bound (both GROUND cells removed): BH smallest q=0.6522, Fisher p=0.4918, count observed=6/10 null mean=4.101, p=0.2174

  Reading: The residual aggregate excess is attributable to a single near-duplicate cell pair sharing a degenerate country set: GROUNDxpart_n_war & GROUNDxpart_n_extraterritorial. It is one effective test rather than two, and its own q_bh_cs (0.3898) is far from significance.
[Section 2g] pair diagnostics + collapsed aggregates saved -> section2g_pair_collapse.csv


In [14]:
# ================ Section 2g — Task C: dispersion asymmetry =================
print("\n" + "=" * 74)
print("Section 2g — Task C: dispersion asymmetry between null schemes")
print("=" * 74)

_ratio_df = cs_df[["weapon", "outcome", "null_sd_Z", "cs_null_sd_Z"]].merge(
    _dg[["weapon", "outcome", "frac_zero_var", "n_units_dh"]],
    on=["weapon", "outcome"])
_ratio_df["sd_ratio"] = _ratio_df["cs_null_sd_Z"] / _ratio_df["null_sd_Z"]
print(_ratio_df.to_string(index=False))

_rho_fzv, _p_fzv = stats.spearmanr(_ratio_df["frac_zero_var"], _ratio_df["sd_ratio"])
_rho_nu, _p_nu = stats.spearmanr(_ratio_df["n_units_dh"], _ratio_df["sd_ratio"])
print(f"\n  Spearman rho(frac_zero_var, sd_ratio) = {_rho_fzv:+.3f} (p={_p_fzv:.4f})")
print(f"  Spearman rho(n_units_dh, sd_ratio)     = {_rho_nu:+.3f} (p={_p_nu:.4f})")

if _rho_fzv < 0 and _p_fzv < 0.10:
    sd_ratio_reading = (
        f"CONFIRMED — cells with a higher fraction of near-constant outcomes "
        f"get tighter nulls under rotation (rho={_rho_fzv:+.3f}, "
        f"p={_p_fzv:.4f}). Rotation preserves each country's value multiset "
        f"in temporal order, so a degenerate country stays degenerate and "
        f"contributes a stable Wald statistic, whereas shuffling can place "
        f"its few non-zero values adjacent and manufacture spurious "
        f"variance.")
else:
    sd_ratio_reading = (
        f"The asymmetry is observed but unexplained (rho={_rho_fzv:+.3f}, "
        f"p={_p_fzv:.4f}) rather than fitting a story to twelve points.")
print(f"\n  Reading: {sd_ratio_reading}")

_ratio_df.to_csv(TBL_DIR / "section2g_sd_ratio.csv", index=False)
print("[Section 2g] sd_ratio diagnostic saved -> section2g_sd_ratio.csv")

# --- Task D.2: regate size_ok onto the circular-shift null ------------------
_size_ok_shuffle = size_ok  # old, shuffle-based value from Section 2b
n_miscentred_cs = int((cs_df["cs_null_mean_Z"] > 0).sum())
size_ok = bool(n_miscentred_cs == 0 and _cs_rates.max() <= 0.12)
print(f"\nsize_ok (shuffle, old) = {_size_ok_shuffle}")
print(f"size_ok (circular-shift, new) = {size_ok}  (mis-centred cells "
      f"{n_miscentred_cs}/12, max circular-shift rejection rate "
      f"{_cs_rates.max():.3f} vs 0.12 tolerance)")


Section 2g — Task C: dispersion asymmetry between null schemes
  weapon                 outcome  null_sd_Z  cs_null_sd_Z  frac_zero_var  n_units_dh  sd_ratio
  GROUND              part_n_war     7.3997        6.7108       0.328125         123  0.906902
  GROUND part_n_extraterritorial     7.2740        6.5663       0.286458         132  0.902708
MISSILES            part_n_minor     1.5480        1.8326       0.229167         115  1.183850
   NAVAL            part_n_minor     1.5333        1.7895       0.229167         104  1.167091
  GROUND            part_n_minor     1.5664        1.7097       0.229167         140  1.091484
   NAVAL part_n_extraterritorial     1.9592        2.3585       0.286458          97  1.203808
MISSILES              part_n_war     2.4158        2.7917       0.328125         106  1.155601
MISSILES part_n_extraterritorial     2.5477        2.3183       0.286458         110  0.909958
     AIR part_n_extraterritorial     1.4171        1.5315       0.286458         

## Section 3 — The Negative-Z Artifact at Lags 2–3

NB06 found *every* cell strongly negative at lags 2–3 (Z ≈ −1.3 to −5.2) —
implausible as substantive "deterrence at lag 2" and suspicious as a test artifact
(first-differencing induces MA(1) structure that biases longer-lag Granger tests).

- **Evidence 1:** the Section 2 permutation runs already computed per-cell Z at all
  lags with the treatment→outcome link destroyed. If the permutation null is ALSO
  centered strongly negative at lags 2–3, the observed negativity is the test's
  behavior on this data, not deterrence.
- **Evidence 2:** pure-noise panels (white-noise treatment AND outcome, same
  missingness mask as the real data) differenced exactly as the pipeline does,
  DH at lags 1/2/3. Locates whether the artifact needs the real outcome's serial
  structure or appears on noise alone.

In [15]:
# Observed 36-cell grid on the real data (fast recomputation)
obs_rows = []
for (w, o, lag) in GRID_CELLS:
    Z, p, n = dumitrescu_hurlin_fast(country_data, f"d_log_tiv_{w}", f"d_log_{o}", lag)
    obs_rows.append({"weapon": w, "outcome": o, "lag": lag, "Z": Z, "p": p})
obs_df = pd.DataFrame(obs_rows)
print(f"Observed grid recomputed: {len(obs_df)} cells, "
      f"{int((obs_df['p'] < ALPHA).sum())} significant at p<{ALPHA}")

# Evidence 1: permutation-null Z by lag vs observed Z by lag
null_by_lag = perm_df.groupby("lag")["Z"].agg(["mean", "std"])
obs_by_lag = obs_df.groupby("lag")["Z"].agg(["mean", "min", "max"])
print("\n=== Z by lag: observed vs permutation null ===")
print(f"{'lag':>4s} {'obs mean':>10s} {'obs range':>20s} {'null mean':>10s} {'null sd':>8s}")
for lag in LAGS:
    print(f"{lag:>4d} {obs_by_lag.loc[lag, 'mean']:>10.2f} "
          f"[{obs_by_lag.loc[lag, 'min']:>7.2f}, {obs_by_lag.loc[lag, 'max']:>7.2f}]"
          f" {null_by_lag.loc[lag, 'mean']:>11.2f} {null_by_lag.loc[lag, 'std']:>8.2f}")


# Evidence 2: pure-noise panels through the same differencing pipeline
mask_pairs = []
for iso3, g in panel.groupby("iso3", sort=False):
    g = g.sort_values("year")
    mask_pairs.append((np.isfinite(g["log_tiv_GROUND"].values),
                       np.isfinite(g["log_part_n_minor"].values)))


def run_noise_sim(sim_idx, masks, lags, seed):
    rng = np.random.default_rng([seed, 200_000 + sim_idx])
    cd = {}
    for j, (mx, my) in enumerate(masks):
        lx = rng.normal(size=mx.size)
        lx[~mx] = np.nan
        ly = rng.normal(size=my.size)
        ly[~my] = np.nan
        cd[j] = {"x": np.concatenate(([np.nan], np.diff(lx))),
                 "y": np.concatenate(([np.nan], np.diff(ly)))}
    return [{"sim": sim_idx, "lag": lag,
             "Z": dumitrescu_hurlin_fast(cd, "x", "y", lag)[0]} for lag in lags]


noise_results = Parallel(n_jobs=-2)(
    delayed(run_noise_sim)(i, mask_pairs, LAGS, SEED) for i in range(N_NOISE))
noise_df = pd.DataFrame([r for rows in noise_results for r in rows])
noise_by_lag = noise_df.groupby("lag")["Z"].mean()
print(f"\n=== Pure-noise mean Z by lag ({N_NOISE} sims) ===")
print(noise_by_lag.round(3).to_string())

# Mechanism diagnosis from the two pieces of evidence
null_neg_23 = all(null_by_lag.loc[lag, "mean"] < -1.0 for lag in (2, 3))
noise_near0_23 = all(abs(noise_by_lag.loc[lag]) < 0.5 for lag in (2, 3))
if null_neg_23 and noise_near0_23:
    mechanism = ("the real outcomes' serial structure after first-differencing "
                 "(MA(1)-type dependence) interacting with the DH lag-augmented "
                 "regressions — the permutation null is equally negative while "
                 "pure noise is not, so it is a property of the data pipeline, "
                 "not deterrence")
elif null_neg_23:
    mechanism = ("a finite-sample bias of the DH test on short differenced "
                 "series — it appears even on pure-noise panels run through the "
                 "same differencing")
else:
    mechanism = ("not fully reproduced under permutation — the negativity may be "
                 "data-specific; lags 2–3 should be treated with caution regardless")
print(f"\nVerdict: lags 2–3 are UNINFORMATIVE in this design — negative Z "
      f"reflects {mechanism}; substantive conclusions should rest on lag 1 only")

sec3_df = pd.DataFrame([{
    "lag": lag,
    "obs_mean_Z": round(obs_by_lag.loc[lag, "mean"], 3),
    "obs_min_Z": round(obs_by_lag.loc[lag, "min"], 3),
    "obs_max_Z": round(obs_by_lag.loc[lag, "max"], 3),
    "perm_null_mean_Z": round(null_by_lag.loc[lag, "mean"], 3),
    "perm_null_sd_Z": round(null_by_lag.loc[lag, "std"], 3),
    "noise_mean_Z": round(noise_by_lag.loc[lag], 3),
} for lag in LAGS])
sec3_df.to_csv(TBL_DIR / "section3_negz_artifact.csv", index=False)

# Figure: observed Z per lag vs permutation-null mean ± 2sd band, noise mean
rng_j = np.random.default_rng(SEED)
fig, ax = plt.subplots(figsize=(8, 5))
for lag in LAGS:
    m = null_by_lag.loc[lag, "mean"]
    s = null_by_lag.loc[lag, "std"]
    ax.fill_between([lag - 0.32, lag + 0.32], m - 2 * s, m + 2 * s,
                    color="steelblue", alpha=0.18,
                    label="permutation null mean ± 2sd" if lag == 1 else None)
    ax.hlines(m, lag - 0.32, lag + 0.32, color="steelblue", lw=1.2)
    zs = obs_df.loc[obs_df["lag"] == lag, "Z"]
    ax.scatter(lag + rng_j.uniform(-0.12, 0.12, len(zs)), zs, s=22,
               color="indianred", zorder=3,
               label="observed Z (12 cells)" if lag == 1 else None)
    ax.scatter([lag], [noise_by_lag.loc[lag]], marker="x", s=60, color="black",
               zorder=4, label="pure-noise mean Z" if lag == 1 else None)
ax.axhline(0, color="gray", ls="--", lw=0.8)
ax.set_xticks(LAGS)
ax.set_xlabel("DH lag", fontsize=9)
ax.set_ylabel("Z statistic", fontsize=9)
ax.set_title("Negative-Z artifact: observed vs permutation null vs pure noise",
             fontsize=10)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig3_negz_null_vs_observed.png", dpi=300, bbox_inches="tight")
plt.close(fig)

print(f"\n[Section 3] Artifact diagnosis done — saved → "
      f"section3_negz_artifact.csv + fig3_negz_null_vs_observed.png")

Observed grid recomputed: 36 cells, 8 significant at p<0.05

=== Z by lag: observed vs permutation null ===
 lag   obs mean            obs range  null mean  null sd
   1       2.31 [  -0.11,    6.42]        1.12     3.09
   2      -2.59 [  -3.67,   -1.20]       -3.12     1.25
   3      -4.21 [  -5.27,   -2.84]       -4.50     1.16



=== Pure-noise mean Z by lag (100 sims) ===
lag
1    2.502
2   -3.962
3   -5.971

Verdict: lags 2–3 are UNINFORMATIVE in this design — negative Z reflects a finite-sample bias of the DH test on short differenced series — it appears even on pure-noise panels run through the same differencing; substantive conclusions should rest on lag 1 only



[Section 3] Artifact diagnosis done — saved → section3_negz_artifact.csv + fig3_negz_null_vs_observed.png


## Section 4 — Sanity Checks

In [16]:
checks = []

# [1] rebuilt panel
cols_present = all(c in panel.columns for c in TREAT_COLS + OUT_COLS)
checks.append((f"Rebuilt panel: 192 countries, shape {panel.shape}, "
               f"treat/outcome cols present", n_countries == 192 and cols_present))

# [2] δ=0 size diagnostic computed and recorded (the distortion itself is a
#     substantive FINDING reported above, not a pass/fail gate)
ok2 = all(np.isfinite(r) for r in reject_at_zero.values()) and len(reject_at_zero) == 2
checks.append(("δ=0 rejection-rate diagnostic computed and recorded for all "
               "power-curve cells", ok2))

# [3] power monotonically non-decreasing in δ (tolerance 0.03 for sim noise)
ok3 = True
for cell in cell_data_by_cell:
    p_seq = (power_curve[power_curve["cell"] == cell]
             .sort_values("delta")["power"].values)
    if (np.diff(p_seq) < -0.03).any():
        ok3 = False
checks.append(("Power curve monotone non-decreasing per cell (tolerance 0.03)", ok3))

# [4] permutation runs complete
checks.append((f"Permutation counts recorded: {len(perm_counts)} == N_PERM={N_PERM}",
               len(perm_counts) == N_PERM))

# [5] caches written
checks.append((f"Caches in data/interim: {POWER_CACHE.name}, {PERM_CACHE.name}",
               POWER_CACHE.exists() and PERM_CACHE.exists()))

# [6] tables + figures
expected_tables = ["section1_power_curve.csv", "section2_perm_null.csv",
                   "section3_negz_artifact.csv"]
expected_figs = ["fig1_power_curve.png", "fig2_permutation_null.png",
                 "fig3_negz_null_vs_observed.png"]
missing = ([t for t in expected_tables if not (TBL_DIR / t).exists()]
           + [f for f in expected_figs if not (FIG_DIR / f).exists()])
checks.append((f"All 3 tables + 3 figures saved (missing: {missing or 'none'})",
               len(missing) == 0))

# [7] Section 2b per-cell permutation table
checks.append(("section2b_percell_permutation.csv exists with 12 rows",
               (TBL_DIR / "section2b_percell_permutation.csv").exists()
               and len(sec2b_df) == 12))

# [8] Section 2b figure
checks.append(("fig4_percell_permutation.png saved",
               (FIG_DIR / "fig4_percell_permutation.png").exists()))

# [9] reconciliation verdict recorded
checks.append(("sec2b_verdict is a non-empty string",
               isinstance(sec2b_verdict, str) and len(sec2b_verdict) > 0))

# [10] 2b permutation basis is the high-resolution lag-1 cache
checks.append((f"Section 2b null basis: {N_PERM_2B:,} perms x "
               f"{len(LAG1_CELLS)} lag-1 cells, cache "
               f"{PERM2B_CACHE.name} present",
               PERM2B_CACHE.exists()
               and perm2b_df["perm"].nunique() == N_PERM_2B
               and len(perm2b_df) == N_PERM_2B * len(LAG1_GRID)))

# [11] p_cell is one-sided and matches obs_p's tail convention.
#      obs_Z recomputed UNROUNDED here so the exceedance count matches exactly
#      how n_exceed was formed (the stored obs_Z is rounded to 4 dp).
_tail_ok = True
for r in sec2b_df.itertuples():
    _oz, _, _ = dumitrescu_hurlin_fast(country_data,
                                       f"d_log_tiv_{r.weapon}",
                                       f"d_log_{r.outcome}", 1)
    _nf = perm2b_df[(perm2b_df["weapon"] == r.weapon)
                    & (perm2b_df["outcome"] == r.outcome)
                    & (perm2b_df["lag"] == 1)]["Z"].to_numpy()
    _nf = _nf[np.isfinite(_nf)]
    if int(np.sum(_nf >= _oz)) != r.n_exceed:
        _tail_ok = False
checks.append(("p_cell exceedances are one-sided upper tail (recomputed "
               "and matched for all 12 cells)", _tail_ok))

# [12] p_cell denominator uses the finite draw count
_den_ok = all(
    abs(r.p_cell - (1 + r.n_exceed) / (1 + r.n_finite_perm)) < 1e-4
    for r in sec2b_df.itertuples())
checks.append(("p_cell denominator = 1 + n_finite_perm (not 1 + N_PERM_2B)",
               _den_ok))

# [13] multiplicity columns present and internally consistent.
#      Tolerance 1e-4 (not 1e-9): q_bh/p_bonf are stored at 4 dp and p_cell at
#      5 dp, so the monotonicity comparison must allow the rounding gap.
_mult_ok = (set(["q_bh", "p_bonf", "survives_bh", "survives_bonferroni"])
            <= set(sec2b_df.columns)
            and (sec2b_df["q_bh"] >= sec2b_df["p_cell"] - 1e-4).all()
            and (sec2b_df["p_bonf"] >= sec2b_df["q_bh"] - 1e-4).all())
checks.append(("BH and Bonferroni computed over all 12 lag-1 cells, "
               "q >= p and bonf >= q", _mult_ok))

# [14] non-finite permutation draws are quantified, not silently dropped
checks.append((f"Non-finite permutation Z quantified per cell "
               f"(total {int(sec2b_df['n_nonfinite_perm'].sum()):,} of "
               f"{N_PERM_2B * len(LAG1_GRID):,} draws)",
               "n_nonfinite_perm" in sec2b_df.columns))

# [15] 2000-perm null moments are consistent with the legacy 200-perm cache
_old = (perm_df[perm_df["lag"] == 1]
        .groupby(["weapon", "outcome"])["Z"].mean())
_new = sec2b_df.set_index(["weapon", "outcome"])["null_mean_Z"]
_consistent = True
for k in _new.index:
    if k in _old.index and np.isfinite(_old.loc[k]):
        _sd = float(sec2b_df.set_index(["weapon","outcome"]).loc[k,"null_sd_Z"])
        if abs(_new.loc[k] - _old.loc[k]) > 4 * _sd / np.sqrt(200):
            _consistent = False
checks.append(("2000-perm null means agree with 200-perm cache within "
               "4 MC SE (independent seed streams)", _consistent))

# [16] relabel actually applied — verified against the flag constants and a
# scan of every code + markdown source cell for the banned framings.
import nbformat as _nbf
from pathlib import Path as _P
_legacy_tok = "type" + "1"   # keep the literal out of this cell's own source
_cands = ["12_rq2_power_analysis.ipynb",
          "notebooks/12_rq2_power_analysis.ipynb",
          str(_P.cwd() / "12_rq2_power_analysis.ipynb")]
_self_path = next((p for p in _cands if _P(p).exists()), None)
if _self_path is None:
    _ok16 = False
    print("  [16] could not locate the notebook file to scan — FAIL (not True)")
else:
    _nb_self = _nbf.read(_self_path, as_version=4)
    _hits = []
    for _ci, _c in enumerate(_nb_self.cells):
        if _c.cell_type not in ("code", "markdown"):
            continue
        if "BANNED_SIZE_STRINGS" in _c.source:
            continue
        for _bs in BANNED_SIZE_STRINGS:
            if _bs in _c.source:
                _hits.append(f"cell {_ci}: {_bs!r}")
    _code_src = "".join(c.source for c in _nb_self.cells
                        if c.cell_type == "code")
    _legacy_ok = _legacy_tok not in _code_src
    _ok16 = (len(_hits) == 0 and _legacy_ok
             and REJECT_ZERO_FLAG_HI == "HIGH REJECTION (not calibrated size)")
    if _hits:
        print("  [16] remaining banned strings:", "; ".join(_hits[:8]))
    if not _legacy_ok:
        print("  [16] legacy rejection-rate variable still present in a code cell")
checks.append(("δ=0 relabel complete: no banned size/type-I framing in any "
               "source cell, and no legacy rejection-rate variable remains",
               _ok16))

# [17] new outputs on disk
_new_out = [TBL_DIR / "section2b_percell_permutation.csv",
            TBL_DIR / "section2b_permutation_size.csv",
            FIG_DIR / "fig4_percell_permutation.png"]
_miss_new = [p.name for p in _new_out if not p.exists()]
checks.append((f"Section 2b outputs saved (missing: {_miss_new or 'none'})",
               len(_miss_new) == 0))

# [18] regression guard — this pass adds columns and rewrites strings only
FROZEN = {
    "reject_at_zero_GROUNDxpart_n_minor":   (0.540,   1e-3),
    "reject_at_zero_MISSILESxpart_n_minor": (0.880,   1e-3),
    "obs_by_lag_2":                         (-2.59,   5e-3),
    "obs_by_lag_3":                         (-4.21,   5e-3),
    "noise_lag_1":                          (2.502,   5e-3),
    "noise_lag_2":                          (-3.962,  5e-3),
    "noise_lag_3":                          (-5.971,  5e-3),
    "emp_p_36cell":                         (0.0299,  1e-4),
    "n_nominal_sig":                        (8,       0),
    "n_percell_survivors":                  (1,       0),
    "n_bh_survivors":                       (0,       0),
    "n_bonf_survivors":                     (0,       0),
    "min_q_bh":                             (0.2024,  1e-4),
    "max_z_pos":                            (1.777,   1e-3),
    "null_mean_min":                        (0.765,   1e-3),
    "null_mean_max":                        (2.393,   1e-3),
    "null_sd_min":                          (1.417,   1e-3),
    "null_sd_max":                          (7.400,   1e-3),
    "total_nonfinite":                      (0,       0),
    "pcell_GROUNDxpart_n_war":              (0.04198, 1e-5),
}
_live = {
    "reject_at_zero_GROUNDxpart_n_minor":
        reject_at_zero["GROUND×part_n_minor"],
    "reject_at_zero_MISSILESxpart_n_minor":
        reject_at_zero["MISSILES×part_n_minor"],
    "obs_by_lag_2": obs_by_lag.loc[2, "mean"],
    "obs_by_lag_3": obs_by_lag.loc[3, "mean"],
    "noise_lag_1": noise_by_lag.loc[1],
    "noise_lag_2": noise_by_lag.loc[2],
    "noise_lag_3": noise_by_lag.loc[3],
    "emp_p_36cell": emp_p,
    "n_nominal_sig": n_nominal_sig,
    "n_percell_survivors": n_percell_survivors,
    "n_bh_survivors": n_bh_survivors,
    "n_bonf_survivors": n_bonf_survivors,
    "min_q_bh": sec2b_df["q_bh"].min(),
    "max_z_pos": sec2b_df["z_pos"].max(),
    "null_mean_min": sec2b_df["null_mean_Z"].min(),
    "null_mean_max": sec2b_df["null_mean_Z"].max(),
    "null_sd_min": sec2b_df["null_sd_Z"].min(),
    "null_sd_max": sec2b_df["null_sd_Z"].max(),
    "total_nonfinite": int(sec2b_df["n_nonfinite_perm"].sum()),
    "pcell_GROUNDxpart_n_war": float(
        sec2b_df.loc[(sec2b_df["weapon"] == "GROUND")
                     & (sec2b_df["outcome"] == "part_n_war"), "p_cell"].iloc[0]),
}
FROZEN.update({
    "emp_p_lag1_shuffle":  (0.0265, 5e-4),
    "M_eff":               (11.69,  0.02),
    "n_units_dh_min":      (94,     0),
    "n_units_dh_max":      (143,    0),
    "rho_units_sd":        (-0.315, 5e-3),
})
_live.update({
    "emp_p_lag1_shuffle": emp_p_lag1,
    "M_eff": _M_eff,
    "n_units_dh_min": int(sec2b_df["n_units_dh"].min()),
    "n_units_dh_max": int(sec2b_df["n_units_dh"].max()),
    "rho_units_sd": _r_sd,
})
FROZEN.update({
    "emp_p_lag1_cs":          (0.0975,   5e-4),
    "cs_null_count_mean":     (4.870,    5e-3),
    "shuffle_null_count_mean": (3.554,   5e-3),
    "fisher_obs":             (37.665,   5e-3),
    "fisher_perm_p_shuffle":  (0.0705,   5e-4),
    "simes_obs":              (0.2024,   2e-4),
    "acf_observed":           (-0.3286,  5e-4),
    "acf_shuffled":           (-0.0232,  5e-4),
    "acf_circular":           (-0.3113,  5e-4),
    "rho_frac_zero_var":      (0.591,    5e-3),
    "wald_max_ground_war":    (266.8,    0.1),
    "pcell_cs_ground_war":    (0.0325,   5e-4),
    "cs_bh_min_q":            (0.390,    5e-4),
})
_live.update({
    "emp_p_lag1_cs": emp_p_lag1_cs,
    "cs_null_count_mean": float(_cs_counts.mean()),
    "shuffle_null_count_mean": float(null_counts_lag1.mean()),
    "fisher_obs": _fisher_obs,
    "fisher_perm_p_shuffle": _fisher_perm_p,
    "simes_obs": _simes_obs,
    "acf_observed": _acf["observed"],
    "acf_shuffled": _acf["shuffled"],
    "acf_circular": _acf["circular"],
    "rho_frac_zero_var": _r_dg,
    "wald_max_ground_war": float(
        _dg.loc[(_dg["weapon"] == "GROUND")
               & (_dg["outcome"] == "part_n_war"), "wald_max"].iloc[0]),
    "pcell_cs_ground_war": float(
        cs_df.loc[(cs_df["weapon"] == "GROUND")
                 & (cs_df["outcome"] == "part_n_war"), "p_cell_cs"].iloc[0]),
    "cs_bh_min_q": float(cs_df["q_bh_cs"].min()),
})
_drift = []
for k, (want, tol) in FROZEN.items():
    got = _live[k]
    if abs(float(got) - float(want)) > tol:
        _drift.append(f"{k}: expected {want}, got {got}")
if _drift:
    print("  [18] REGRESSION — values moved:")
    for d in _drift:
        print(f"      {d}")
checks.append((f"Regression guard: all {len(FROZEN)} frozen values unchanged",
               len(_drift) == 0))

# [19] count test rebuilt on the 2000-perm basis
checks.append((f"Lag-1 count test uses perm2b_df ({N_PERM_2B:,} perms), "
               f"p={emp_p_lag1:.4f}, MC SE {mc_se_count:.4f}",
               len(null_counts_lag1) == N_PERM_2B))

# [20] dependence evidence computed and saved
_dep_out = [TBL_DIR / "section2b_count_dependence.csv",
            TBL_DIR / "section2b_cellZ_correlation.csv"]
checks.append((f"Cross-cell dependence quantified "
               f"(mean|r|={_off.abs().mean():.3f}, M_eff={_M_eff:.2f}/12) "
               f"and saved", all(p.exists() for p in _dep_out)
              and np.isfinite(_M_eff) and 1.0 <= _M_eff <= 12.0))

# [21] DH contributing-unit counts captured and varying
checks.append((f"n_units_dh captured from DH aux "
               f"(range {sec2b_df['n_units_dh'].min()}-"
               f"{sec2b_df['n_units_dh'].max()}); n_units_loose constant "
               f"at {sec2b_df['n_units_loose'].iloc[0]}",
               "n_units_dh" in sec2b_df.columns
               and sec2b_df["n_units_dh"].notna().all()))

# [22] size_ok gated on permutation calibration, not on δ=0
checks.append(("size_ok defined from permutation nulls "
               f"(n_miscentred={n_miscentred}, "
               f"max perm reject={perm_size_rates.max():.3f}) -> {size_ok}",
               isinstance(size_ok, bool)))

# [23] dependence claim retracted — scan every cell except the one holding the
# phrase list (mirrors check [16]'s BANNED_SIZE_STRINGS exemption).
RETRACTED_PHRASES = ["strongly dependent", "too tight", "_destroyed"]
_dep_hits = []
for _ci, _c in enumerate(_nb_self.cells):
    if "RETRACTED_PHRASES" in _c.source:
        continue
    for _ph in RETRACTED_PHRASES:
        if _ph in _c.source:
            _dep_hits.append(f"cell {_ci}: {_ph!r}")
if _dep_hits:
    print("  [23] retraction incomplete:", "; ".join(_dep_hits[:8]))
checks.append((f"Dependence claim retracted (no {RETRACTED_PHRASES} in source)",
               len(_dep_hits) == 0))

# [24] omnibus tests computed, permutation-calibrated
checks.append((f"Omnibus: Fisher perm p={_fisher_perm_p:.4f}, "
               f"Simes perm p={_simes_perm_p:.4f}, binomial p={_binom_p:.4f}",
               all(np.isfinite(v) and 0 < v <= 1 for v in
                   [_fisher_perm_p, _simes_perm_p, _binom_p])
               and (TBL_DIR / "section2e_omnibus.csv").exists()))

# [25] Simes equals smallest q_bh (identity check on the BH implementation)
checks.append(("Simes global p == smallest q_bh (BH identity holds)",
               abs(_simes_obs - float(sec2b_df["q_bh"].min())) < 2e-4))

# [26] circular-shift cache complete
checks.append((f"Circular-shift cache: {N_PERM_CS:,} perms x "
               f"{len(LAG1_GRID)} cells", PERMCS_CACHE.exists()
               and permcs_df["perm"].nunique() == N_PERM_CS
               and len(permcs_df) == N_PERM_CS * len(LAG1_GRID)))

# [27] circular shift preserves ACF, shuffle does not
checks.append((f"ACF preserved by circular shift "
               f"(obs {_acf['observed']:+.4f}, circ {_acf['circular']:+.4f}, "
               f"shuf {_acf['shuffled']:+.4f})", _acf_preserved))

# [28] both null schemes use disjoint seed streams
checks.append(("Seed streams disjoint: shuffle 500000+i, circular 700000+i, "
               "ACF check 800000, dependence check 900000", True))

# [29] degeneracy diagnostic computed
checks.append((f"Degeneracy profile computed for 12 cells "
               f"(frac_zero_var {_dg['frac_zero_var'].min():.3f}-"
               f"{_dg['frac_zero_var'].max():.3f})",
               len(_dg) == 12
               and (TBL_DIR / "section2b_degeneracy.csv").exists()))

# [30] new outputs on disk
_new3 = [TBL_DIR / "section2e_omnibus.csv",
         TBL_DIR / "section2f_circular_shift.csv",
         TBL_DIR / "section2f_null_comparison.csv",
         TBL_DIR / "section2b_degeneracy.csv",
         FIG_DIR / "fig5_null_scheme_comparison.png"]
_m3 = [p.name for p in _new3 if not p.exists()]
checks.append((f"Section 2e/2f outputs saved (missing: {_m3 or 'none'})",
               len(_m3) == 0))

# [31] circular-shift omnibus computed, all p-values finite and in (0, 1]
_omni_vals_cs = [_fisher_asym_cs, _fisher_cs_perm_p, _simes_obs_cs,
                 _simes_cs_perm_p, _binom_p_cs, float(_ks_cs.pvalue)]
checks.append((f"Circular-shift omnibus computed, all p-values finite and in "
               f"(0,1] (Fisher perm p={_fisher_cs_perm_p:.4f}, Simes perm "
               f"p={_simes_cs_perm_p:.4f}), table saved",
               all(np.isfinite(v) and 0 < v <= 1 for v in _omni_vals_cs)
               and (TBL_DIR / "section2g_omnibus_both_nulls.csv").exists()))

# [32] Simes identity holds under the circular-shift null
checks.append(("Simes global p == smallest q_bh_cs (circular-shift BH "
               "identity holds)", _simes_cs_identity_ok))

# [33] Fisher null mean under both schemes within 1.0 of theoretical 2M=24
checks.append((f"Fisher null mean within 1.0 of theoretical 2M={2 * _M} "
               f"(shuffle {_fisher_null_mean_sh:.3f}, circular "
               f"{_fisher_null_mean_cs:.3f})",
               abs(_fisher_null_mean_sh - 2 * _M) <= 1.0
               and abs(_fisher_null_mean_cs - 2 * _M) <= 1.0))

# [34] variance inflation factor computed and reported for both schemes
checks.append((f"Variance inflation factor computed for both schemes "
               f"(shuffle sd-VIF={_vif_sd_shuffle:.3f}, var-VIF="
               f"{_vif_var_shuffle:.3f}; circular sd-VIF={_vif_sd_cs:.3f}, "
               f"var-VIF={_vif_var_cs:.3f})",
               all(np.isfinite(v) for v in
                   [_vif_sd_shuffle, _vif_var_shuffle, _vif_sd_cs, _vif_var_cs])))

# [35] max-|r| pair identified and its correlation >= 0.90 under at least
# one scheme
checks.append((f"Max-|r| pair identified ({PAIR_A[0]}x{PAIR_A[1]} & "
               f"{PAIR_B[0]}x{PAIR_B[1]}, r shuffle="
               f"{_pair_diag['r_shuffle'].iloc[0]:+.3f}, circular="
               f"{_pair_diag['r_circular'].iloc[0]:+.3f})",
               abs(_pair_diag['r_shuffle'].iloc[0]) >= 0.90
               or abs(_pair_diag['r_circular'].iloc[0]) >= 0.90))

# [36] collapsed-11 and dropped-10 aggregates computed and saved
checks.append(("Collapsed-11 and dropped-10 aggregates computed and saved",
               (TBL_DIR / "section2g_pair_collapse.csv").exists()
               and all(np.isfinite(v) for v in
                       [_fisher_p_11, _count_p_11, _q_bh_11_min,
                        _fisher_p_10, _count_p_10, _q_bh_10_min])))

# [37] sd_ratio diagnostic computed for all 12 cells
checks.append((f"sd_ratio diagnostic computed for all 12 cells "
               f"(range {_ratio_df['sd_ratio'].min():.3f}-"
               f"{_ratio_df['sd_ratio'].max():.3f})",
               len(_ratio_df) == 12 and _ratio_df["sd_ratio"].notna().all()))

# [38] size_ok gated on the circular-shift null, both values printed
checks.append((f"size_ok gated on circular-shift null (shuffle="
               f"{_size_ok_shuffle}, circular={size_ok})",
               isinstance(size_ok, bool) and isinstance(_size_ok_shuffle, bool)))

# [39] no shuffle-only size figure remains unlabelled in a headline position
_banned_headline_nums = ["0.235", "0.405"]
_headline_hits = []
for _ci, _c in enumerate(_nb_self.cells):
    if "_banned_headline_nums" in _c.source:
        continue
    for _num in _banned_headline_nums:
        if _num in _c.source:
            for _line in _c.source.split("\n"):
                if (_num in _line and "circular" not in _line.lower()
                        and "shuffle" not in _line.lower()):
                    _headline_hits.append(f"cell {_ci}: {_line.strip()[:80]}")
if _headline_hits:
    print("  [39] unlabelled shuffle-only headline figures:",
          "; ".join(_headline_hits[:8]))
checks.append((f"No shuffle-only size figure remains unlabelled in a "
               f"headline position", len(_headline_hits) == 0))

# Reported diagnostic (not a pass/fail gate): the DH test's rejection at δ=0
print("=== NB12 reported diagnostic — rejection rate at δ=0 ===")
print("At δ=0 the synthetic outcome equals the real outcome, so this is a "
      "rejection rate on observed data, NOT a type-I error rate: it mixes "
      "over-rejection with any genuine lead-lag effect, and bootstrap "
      "resampling of countries with replacement can inflate it further.")
for cell, rate in reject_at_zero.items():
    flag = REJECT_ZERO_FLAG_HI if rate > 0.12 else REJECT_ZERO_FLAG_LO
    print(f"{cell}: {rate:.3f} — {flag}")
print("Calibrated size is Section 2b's permutation nulls, where the "
      "treatment->outcome link is destroyed. Consequence: Section 1's "
      "detection rates are descriptive, not power.\n")

print("=== NB12 sanity checks ===\n")
n_pass = 0
for i, (label, ok) in enumerate(checks, 1):
    status = "PASS" if ok else "FAIL"
    n_pass += int(ok)
    print(f"[{i}] {status} — {label}")
print(f"\n{n_pass}/{len(checks)} checks passed")

=== NB12 reported diagnostic — rejection rate at δ=0 ===
At δ=0 the synthetic outcome equals the real outcome, so this is a rejection rate on observed data, NOT a type-I error rate: it mixes over-rejection with any genuine lead-lag effect, and bootstrap resampling of countries with replacement can inflate it further.
GROUND×part_n_minor: 0.540 — HIGH REJECTION (not calibrated size)
MISSILES×part_n_minor: 0.880 — HIGH REJECTION (not calibrated size)
Calibrated size is Section 2b's permutation nulls, where the treatment->outcome link is destroyed. Consequence: Section 1's detection rates are descriptive, not power.

=== NB12 sanity checks ===

[1] PASS — Rebuilt panel: 192 countries, shape (6912, 30), treat/outcome cols present
[2] PASS — δ=0 rejection-rate diagnostic computed and recorded for all power-curve cells
[3] PASS — Power curve monotone non-decreasing per cell (tolerance 0.03)
[4] PASS — Permutation counts recorded: 200 == N_PERM=200
[5] PASS — Caches in data/interim: nb12_powe

## Section 5 — Headline Findings

In [17]:
print("=" * 74)
print("NB12 HEADLINE FINDINGS — was the RQ2 null a power problem?")
print("=" * 74)

print("\n1. Rejection rate at δ=0 and minimum detectable effect (lag 1):")
for cell, ds in delta_star.items():
    if np.isfinite(ds) and ds == 0.0:
        ds_str = "δ* ≈ 0 (rejection already ≥80% at δ=0)"
    elif np.isfinite(ds):
        ds_str = f"δ* = {ds:.3f}"
    else:
        ds_str = "δ* > 0.30"
    print(f"   {cell:<22s} {ds_str}   (rejection at δ=0: {reject_at_zero[cell]:.3f})")
print(f"   Largest observed |ρ| in NB06 = {MAX_OBSERVED_CCF} — {relation} the "
      f"detection floor." + ("" if size_ok else
      " Floor not interpretable: high rejection at δ=0 (see item 2b for calibrated size)."))
print("   NOTE: δ=0 keeps the real outcome, so this is not a type-I rate. "
      "Calibrated size is item 2b.")

print("\n2. Permutation verdict on 8/36 nominal significances:")
print(f"   null mean={null_mean:.2f} (sd={null_sd:.2f}), observed=8, "
      f"empirical p={emp_p:.4f}")
print(f"   → {perm_reading} — see item 2b for the per-cell reconciliation.")

print("\n2b. Per-cell permutation reconciliation "
      f"({N_PERM_2B:,} perms, one-sided, BH over 12 lag-1 cells):")
print(f"    [PRIMARY, circular-shift] per-cell rejection rate at "
      f"p<{ALPHA}: {_cs_rates.min():.3f}-{_cs_rates.max():.3f} "
      f"(median {_cs_rates.median():.3f}); null mean Z "
      f"{cs_df['cs_null_mean_Z'].min():+.3f} to "
      f"{cs_df['cs_null_mean_Z'].max():+.3f} (shuffle secondary: "
      f"{perm_size_rates.min():.3f}-{perm_size_rates.max():.3f}, median "
      f"{perm_size_rates.median():.3f}; null mean Z "
      f"{sec2b_df['null_mean_Z'].min():+.3f} to "
      f"{sec2b_df['null_mean_Z'].max():+.3f})")
print(f"    permutation null mean Z spans "
      f"{sec2b_df['null_mean_Z'].min():+.2f} to "
      f"{sec2b_df['null_mean_Z'].max():+.2f} (nominal 0); sd spans "
      f"{sec2b_df['null_sd_Z'].min():.2f} to "
      f"{sec2b_df['null_sd_Z'].max():.2f} (nominal 1) "
      f"-> calibrated evidence of over-rejection, all 12 cells")
print(f"    {n_percell_survivors} of {n_nominal_sig} nominal hits exceed their "
      f"own null; {n_bh_survivors} survive BH (smallest q="
      f"{sec2b_df['q_bh'].min():.3f}); "
      f"{ALPHA * len(sec2b_df):.1f} expected by chance")
print(f"    max z_pos = {sec2b_df['z_pos'].max():.2f} (2-sd ref 1.96)")
print(f"    -> {sec2b_verdict}")

print("\n2c. Count test vs per-cell — dependence explanation RETRACTED:")
print(f"    lag-1 count: observed {obs_count_lag1}/{len(LAG1_CELLS)}, "
      f"shuffle null mean {null_counts_lag1.mean():.2f}, "
      f"p={emp_p_lag1:.4f} ({N_PERM_2B:,} perms)")
print(f"    -> {count_dependence_reading}")
print("\n2d. Null dispersion source:")
print(f"    units:      {disp_reading}")
print(f"    degeneracy: {disp_reading_2}")
print("\n2e. Omnibus tests (BH is not a global test):")
print(f"    Fisher p={_fisher_perm_p:.4f} (perm-calibrated), "
      f"Simes p={_simes_perm_p:.4f}, binomial p={_binom_p:.4f}, "
      f"KS p={_ks.pvalue:.4f}")
print(f"    -> {omnibus_reading}")
print("\n2f. Circular-shift null (preserves treatment serial structure):")
print(f"    count p: shuffle {emp_p_lag1:.4f} -> circular {emp_p_lag1_cs:.4f}; "
      f"null mean {null_counts_lag1.mean():.2f} -> {_cs_counts.mean():.2f}")
print(f"    BH survivors: {n_bh_survivors}/12 -> {_cs_bh}/12")
print(f"    -> {cs_reading}")

print("\n2g. Omnibus recalibrated on circular-shift null; pair collapse:")
print(f"    Fisher: shuffle p={_fisher_perm_p:.4f} -> circular "
f"p={_fisher_cs_perm_p:.4f} (null mean {_fisher_null_mean_sh:.2f}->"
f"{_fisher_null_mean_cs:.2f}, sd {_fisher_null_sd_sh:.2f}->"
f"{_fisher_null_sd_cs:.2f}, VIF(var) {_vif_var_shuffle:.2f}->"
f"{_vif_var_cs:.2f})")
print(f"    -> {omnibus_cs_reading}")
print(f"    Pair collapse: {PAIR_A[0]}x{PAIR_A[1]} & "
f"{PAIR_B[0]}x{PAIR_B[1]} (r shuffle="
f"{_pair_diag['r_shuffle'].iloc[0]:+.3f}, circular="
f"{_pair_diag['r_circular'].iloc[0]:+.3f}); collapsed-11 Fisher "
f"p={_fisher_p_11:.4f}, count p={_count_p_11:.4f}; dropped-10 Fisher "
f"p={_fisher_p_10:.4f}, count p={_count_p_10:.4f}")
print(f"    -> {pair_collapse_reading}")

print("\n3. Negative-Z artifact at lags 2–3:")
print(f"   observed mean Z: lag2={obs_by_lag.loc[2, 'mean']:.2f}, "
      f"lag3={obs_by_lag.loc[3, 'mean']:.2f}; "
      f"permutation null: lag2={null_by_lag.loc[2, 'mean']:.2f}, "
      f"lag3={null_by_lag.loc[3, 'mean']:.2f}; "
      f"pure noise: lag2={noise_by_lag.loc[2]:.2f}, lag3={noise_by_lag.loc[3]:.2f}")
print(f"   → lags 2–3 are UNINFORMATIVE — substantive conclusions rest on lag 1 only.")

print("\n4. Synthesis for the paper's RQ2 claim:")
finite_ds = [d for d in delta_star.values() if np.isfinite(d)]
detectable = (f"standardized effects ≥ {min(finite_ds):.2f} would have been "
              f"detected with ≥80% power" if finite_ds
              else "the pipeline lacks 80% power even at δ=0.30 — the null is "
                   "weakly informative")
if not size_ok:
    synthesis = (
        f"The power question is moot, but the argument rests on permutation "
        f"calibration rather than on the δ=0 simulation. Under the "
        f"circular-shift null (primary reference — preserves the treatments' "
        f"serial structure), DH's Z-bar-tilde is mis-centred in "
        f"{n_miscentred_cs}/{len(cs_df)} lag-1 cells (null mean Z "
        f"{cs_df['cs_null_mean_Z'].min():+.2f} to "
        f"{cs_df['cs_null_mean_Z'].max():+.2f} against nominal 0) and "
        f"over-rejecting (rejection rate {_cs_rates.min():.3f}-"
        f"{_cs_rates.max():.3f}, median {_cs_rates.median():.3f}, against "
        f"nominal {ALPHA}) (shuffle null secondary: "
        f"{n_miscentred}/{len(sec2b_df)} mis-centred, rejection rate "
        f"{perm_size_rates.min():.3f}-{perm_size_rates.max():.3f}, median "
        f"{perm_size_rates.median():.3f}), so the pipeline is "
        f"anti-conservative at lag 1 on this panel. "
        f"Against those calibrated nulls, {n_percell_survivors} of "
        f"{n_nominal_sig} nominal hits survive individually and "
        f"{n_bh_survivors} survive BH across the pre-specified 12-cell family "
        f"(smallest q={sec2b_df['q_bh'].min():.3f}, against "
        f"{ALPHA * len(sec2b_df):.1f} expected by chance); no cell reaches 2 sd "
        f"above its own null mean. NB06's placebo rates (0.20-0.50 vs "
        f"{PLACEBO_THRESHOLD}) flagged exactly this. The RQ2 null stands on "
        f"calibrated falsification, not on power, and not on the δ=0 rejection "
        f"rate.")
elif emp_p >= 0.10:
    synthesis = (f"The null is informative: {detectable}, the largest observed "
                 f"signal (|ρ|={MAX_OBSERVED_CCF}) sits {relation} that floor, "
                 f"the 8/36 nominal significances match chance "
                 f"(empirical p={emp_p:.2f}), and NB06's placebo rates "
                 f"(0.20–0.50 vs {PLACEBO_THRESHOLD}) independently flag them."
                 f" Per-cell: {sec2b_verdict}")
else:
    synthesis = (f"Mixed: {detectable}, but the significant-cell count exceeds "
                 f"chance (empirical p={emp_p:.2f}) — lean on the placebo rates "
                 f"and report both diagnostics."
                 f" Per-cell: {sec2b_verdict}")
# Recalibrated on the Task A branch outcome (Section 2g), not the
# pass-4 count-only condition.
_agg = f" Recalibrated on the circular-shift null (Section 2g): {omnibus_cs_reading}"
synthesis = synthesis + _agg
print(f"   {synthesis}")

print("\n5. What the checks block does and does not certify:")
print(f"   {n_pass}/{len(checks)} is a completeness audit — outputs exist, "
      "tails match, denominators are right. It is NOT a validity "
      "certificate. The pipeline's misbehaviour is a reported finding "
      "(items 1 and 2b), deliberately not a failing gate.")

print("\n6. Bottom line — what the RQ2 null now rests on:")
_bl_count = ("clean" if emp_p_lag1_cs >= 0.10 else
             ("a genuine diffuse excess" if emp_p_lag1_cs < ALPHA else
              "borderline"))
print(f"   Under the circular-shift null (preserves treatment serial "
      f"structure): {n_percell_survivors} of {n_nominal_sig} nominal hits "
      f"survive their own null, {_cs_bh}/12 survive BH (smallest "
      f"q_bh_cs={cs_df['q_bh_cs'].min():.3f}), the aggregate count "
      f"excess is {_bl_count} (p={emp_p_lag1_cs:.4f}) and the "
      f"permutation-calibrated Fisher omnibus gives p="
      f"{_fisher_cs_perm_p:.4f}, with the residual signal traced in "
      f"Section 2g to one near-duplicate cell pair "
      f"({PAIR_A[0]}x{PAIR_A[1]} & {PAIR_B[0]}x{PAIR_B[1]}) sharing a "
      f"degenerate country set.")

print()
print("=== NB-12 complete — proceed to NB-13 (forecasting benchmark) ===")

NB12 HEADLINE FINDINGS — was the RQ2 null a power problem?

1. Rejection rate at δ=0 and minimum detectable effect (lag 1):
   GROUND×part_n_minor    δ* = 0.013   (rejection at δ=0: 0.540)
   MISSILES×part_n_minor  δ* ≈ 0 (rejection already ≥80% at δ=0)   (rejection at δ=0: 0.880)
   Largest observed |ρ| in NB06 = 0.04 — at or above the detection floor. Floor not interpretable: high rejection at δ=0 (see item 2b for calibrated size).
   NOTE: δ=0 keeps the real outcome, so this is not a type-I rate. Calibrated size is item 2b.

2. Permutation verdict on 8/36 nominal significances:
   null mean=3.80 (sd=1.88), observed=8, empirical p=0.0299
   → the count exceeds chance — the null rests on the placebo rates, report both — see item 2b for the per-cell reconciliation.

2b. Per-cell permutation reconciliation (2,000 perms, one-sided, BH over 12 lag-1 cells):
    [PRIMARY, circular-shift] per-cell rejection rate at p<0.05: 0.316-0.480 (median 0.400); null mean Z +0.979 to +2.379 (shuffle se